<a href="https://colab.research.google.com/github/Miguithub/NPP_Cripto_5min/blob/Experimentos-112024-012026/Eth%3B_Ectx%3B_Libre_112024_012026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd

Mounted at /content/drive


In [ ]:
ruta = "/content/drive/MyDrive/Neural/NPP/Cripto/Crypto 5 minutos/"

df1_ctx = pd.read_parquet(ruta + "112024-042025_clean_add_final_Ectx.parquet")
df2_ctx = pd.read_parquet(ruta + "042025-092025_clean_add_final_Ectx.parquet")
df3_ctx = pd.read_parquet(ruta + "092025-022026_clean_add_final_Ectx.parquet")

In [ ]:
df1_ctx.info()
df2_ctx.info()
df3_ctx.info()

In [ ]:
ruta = "/content/drive/MyDrive/Neural/NPP/Cripto/Crypto 5 minutos/"

df1_th = pd.read_parquet(ruta + "112024-042025_clean_add_final_Eth.parquet")
df2_th = pd.read_parquet(ruta + "042025-092025_clean_add_final_Eth.parquet")
df3_th = pd.read_parquet(ruta + "092025-022026_clean_add_final_Eth.parquet")

In [ ]:
df1_th.info()
df2_th.info()
df3_th.info()

# Drive segmentado

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd

Mounted at /content/drive


In [2]:
ruta = "/content/drive/MyDrive/Neural/NPP/Cripto/Crypto 5 minutos/"

df1_ctx = pd.read_parquet(ruta + "112024-042025_clean_add_final_Ectx.parquet")

In [3]:
df1_ctx.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43775 entries, 0 to 43774
Columns: 35708 entries, bar_end to 币安人生__Ectx_ERANK
dtypes: datetime64[ns, UTC](1), float32(34687), int8(1020)
memory usage: 5.7 GB


In [4]:
ruta = "/content/drive/MyDrive/Neural/NPP/Cripto/Crypto 5 minutos/"

df1_th = pd.read_parquet(ruta + "112024-042025_clean_add_final_Eth.parquet")

In [5]:
df1_th.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43775 entries, 0 to 43774
Columns: 35708 entries, bar_end to 币安人生__Eth_ERANK
dtypes: datetime64[ns, UTC](1), float32(34687), int8(1020)
memory usage: 5.7 GB


#Experimento A: Softmax

## Baseline

In [7]:
# ============================================================
# HELPERS DE CARGA
# ============================================================

def feat_col(asset, feature):
    return f"{asset}__{feature}"


def base_col(asset, kind, cfg):

    if kind == "P":
        return f"{asset}{cfg.price_suffix}"

    suffix = {
        "V": cfg.volume_suffix,
        "C": cfg.trades_suffix,
        "R": cfg.range_suffix,
        "L": cfg.illiq_suffix,
        "A": cfg.active_suffix,
        "O": cfg.observed_suffix,
        "B": cfg.buy_suffix,
        "S": cfg.sell_suffix,
        "yp": cfg.yp_suffix,
    }[kind]

    return f"{asset}{suffix}"


def load_feature(df, assets, feature):

    cols = [
        feat_col(a, feature)
        for a in assets
    ]

    out = df.loc[:, cols].astype(
        "float64",
        copy=False
    )

    out.columns = assets

    return out


def load_base(df, assets, kind, cfg):

    cols = [
        base_col(a, kind, cfg)
        for a in assets
    ]

    out = df.loc[:, cols].astype(
        "float64",
        copy=False
    )

    out.columns = assets

    return out

In [9]:
# ============================================================
# RECONSTRUIR LISTA DE ACTIVOS
# ============================================================

assets = sorted(
    c.removesuffix("__E_context")
    for c in df1_ctx.columns
    if c.endswith("__E_context")
)

print("Activos detectados:", len(assets))


# ============================================================
# VALIDACIÓN DE COLUMNAS
# ============================================================

print(
    "E_context:",
    all(
        f"{a}__E_context" in df1_ctx.columns
        for a in assets
    )
)

print(
    "E_theory:",
    all(
        f"{a}__E_theory" in df1_th.columns
        for a in assets
    )
)

print(
    "yp:",
    all(
        f"{a}_yp" in df1_ctx.columns
        for a in assets
    )
)

Activos detectados: 510
E_context: True
E_theory: True
yp: True


In [10]:
# ============================================================
# VALIDACIÓN CTX vs THEORY
# ============================================================

assets_ctx = {
    c.removesuffix("__E_context")
    for c in df1_ctx.columns
    if c.endswith("__E_context")
}

assets_th = {
    c.removesuffix("__E_theory")
    for c in df1_th.columns
    if c.endswith("__E_theory")
}

print("Activos CTX:", len(assets_ctx))
print("Activos TH:", len(assets_th))
print("Conjuntos idénticos:", assets_ctx == assets_th)

print("Solo CTX:", assets_ctx - assets_th)
print("Solo TH:", assets_th - assets_ctx)

Activos CTX: 510
Activos TH: 510
Conjuntos idénticos: True
Solo CTX: set()
Solo TH: set()


In [13]:
# ============================================================
# CONFIGURACIÓN ORIGINAL DEL PIPELINE
# ============================================================

from dataclasses import dataclass


@dataclass(frozen=True)
class FeatureConfig:

    time_col: str = "bar_end"
    ys_col: str = "YS"

    price_suffix: str = "USDT"
    volume_suffix: str = "_v"
    trades_suffix: str = "_activity_trades_proxy"
    range_suffix: str = "_range_rel_proxy"
    illiq_suffix: str = "_illiquidity_proxy"
    active_suffix: str = "_active_trading"
    observed_suffix: str = "_observed"
    buy_suffix: str = "_taker_buy_usdt_proxy"
    sell_suffix: str = "_taker_sell_usdt_proxy"
    yp_suffix: str = "_yp"

    primary_window: int = 288
    min_periods: int = 72
    ewm_span: int = 72

    multiscale_windows: tuple[int, ...] = (
        3,
        12,
        72,
        288,
    )

    return_lag: int = 1

    stress_quantile: float = 0.90

    recovery_tolerance: float = 0.15
    recovery_hold_bars: int = 3
    recovery_event_ewm_alpha: float = 0.20

    eps: float = 1e-12
    dtype: str = "float32"

    asset_block_size: int = 32


cfg = FeatureConfig()

print(cfg)

FeatureConfig(time_col='bar_end', ys_col='YS', price_suffix='USDT', volume_suffix='_v', trades_suffix='_activity_trades_proxy', range_suffix='_range_rel_proxy', illiq_suffix='_illiquidity_proxy', active_suffix='_active_trading', observed_suffix='_observed', buy_suffix='_taker_buy_usdt_proxy', sell_suffix='_taker_sell_usdt_proxy', yp_suffix='_yp', primary_window=288, min_periods=72, ewm_span=72, multiscale_windows=(3, 12, 72, 288), return_lag=1, stress_quantile=0.9, recovery_tolerance=0.15, recovery_hold_bars=3, recovery_event_ewm_alpha=0.2, eps=1e-12, dtype='float32', asset_block_size=32)


In [14]:
# ============================================================
# VALIDACIÓN DEL ENTORNO
# ============================================================

checks = {
    "df1_ctx": "df1_ctx" in globals(),
    "df1_th": "df1_th" in globals(),
    "assets": "assets" in globals(),
    "cfg": "cfg" in globals(),
    "load_feature": "load_feature" in globals(),
    "load_base": "load_base" in globals(),
}

for name, ok in checks.items():
    print(f"{name:15} {'OK' if ok else 'FALTA'}")

df1_ctx         OK
df1_th          OK
assets          OK
cfg             OK
load_feature    OK
load_base       OK


In [15]:
# ============================================================
# TEST DE CARGA
# ============================================================

E_ctx_test = load_feature(
    df1_ctx,
    assets,
    "E_context"
)

E_th_test = load_feature(
    df1_th,
    assets,
    "E_theory"
)

yp_test = load_base(
    df1_ctx,
    assets,
    "yp",
    cfg
)

A_test = load_base(
    df1_ctx,
    assets,
    "A",
    cfg
)

print("E_context:", E_ctx_test.shape)
print("E_theory :", E_th_test.shape)
print("yp       :", yp_test.shape)
print("A        :", A_test.shape)

E_context: (43775, 510)
E_theory : (43775, 510)
yp       : (43775, 510)
A        : (43775, 510)


##Experimento A

###A1 Predictiva 1

In [16]:
import numpy as np
import pandas as pd

from scipy.optimize import minimize_scalar, minimize


# ============================================================
# CONFIGURACIÓN EXPERIMENTO A
# ============================================================

EPS = 1e-12

TRAIN_FRAC = 0.70
VAL_FRAC = 0.15

CTX_FEATURE = "E_context"
TH_FEATURE = "E_theory"


# ============================================================
# CARGA
# ============================================================

E_ctx = load_feature(
    df1_ctx,
    assets,
    CTX_FEATURE
).to_numpy(dtype=np.float64, copy=False)

E_th = load_feature(
    df1_th,
    assets,
    TH_FEATURE
).to_numpy(dtype=np.float64, copy=False)

yp_t = load_base(
    df1_ctx,
    assets,
    "yp",
    cfg
).to_numpy(dtype=np.float64, copy=False)

A_t = load_base(
    df1_ctx,
    assets,
    "A",
    cfg
).to_numpy(dtype=np.float64, copy=False)


# ============================================================
# TARGET t+1
# ============================================================

Y = np.empty_like(yp_t)

Y[:-1] = yp_t[1:]
Y[-1] = np.nan


# ============================================================
# SOPORTE COMÚN CAUSAL
# Solo activos conocidos/activos en t y con ambas energías válidas
# ============================================================

mask = (
    (A_t > 0)
    & np.isfinite(E_ctx)
    & np.isfinite(E_th)
    & np.isfinite(Y)
)


# ============================================================
# CIERRE AL SIMPLEX
# ============================================================

def closure(x, mask, eps=EPS):

    z = np.where(
        mask,
        np.maximum(x, 0.0),
        0.0
    )

    total = z.sum(
        axis=1,
        keepdims=True
    )

    return np.divide(
        z,
        total,
        out=np.zeros_like(z),
        where=total > eps
    )


Y = closure(
    Y,
    mask
)


# ============================================================
# FILAS VÁLIDAS
# ============================================================

valid_rows = (
    mask.sum(axis=1) >= 2
) & (
    Y.sum(axis=1) > 1 - 1e-8
)

E_ctx = E_ctx[valid_rows]
E_th = E_th[valid_rows]
yp_t_valid = yp_t[valid_rows]
Y = Y[valid_rows]
mask = mask[valid_rows]


# ============================================================
# SPLIT TEMPORAL
# ============================================================

n = len(Y)

i_train = int(
    n * TRAIN_FRAC
)

i_val = int(
    n * (TRAIN_FRAC + VAL_FRAC)
)

splits = {
    "train": slice(0, i_train),
    "val": slice(i_train, i_val),
    "test": slice(i_val, n),
}

print(
    f"Train: {i_train:,} | "
    f"Val: {i_val - i_train:,} | "
    f"Test: {n - i_val:,}"
)


# ============================================================
# SOFTMAX ENERGÉTICO ESTABLE
# ============================================================

def energy_softmax(
    E,
    mask,
    T,
):

    logits = -E / T

    logits = np.where(
        mask,
        logits,
        -np.inf
    )

    row_max = np.max(
        logits,
        axis=1,
        keepdims=True
    )

    exp_logits = np.where(
        mask,
        np.exp(logits - row_max),
        0.0
    )

    denominator = exp_logits.sum(
        axis=1,
        keepdims=True
    )

    return np.divide(
        exp_logits,
        denominator,
        out=np.zeros_like(exp_logits),
        where=denominator > 0
    )


# ============================================================
# CROSS ENTROPY
# ============================================================

def cross_entropy(
    Y,
    P,
    mask,
    eps=EPS,
):

    return -np.sum(
        np.where(
            mask,
            Y * np.log(P + eps),
            0.0
        )
    ) / len(Y)


# ============================================================
# AJUSTE TEMPERATURA
# ============================================================

def fit_temperature(
    E,
    Y,
    mask,
):

    def objective(log_T):

        T = np.exp(log_T)

        P = energy_softmax(
            E,
            mask,
            T
        )

        return cross_entropy(
            Y,
            P,
            mask
        )

    result = minimize_scalar(
        objective,
        bounds=(-8, 8),
        method="bounded"
    )

    return np.exp(
        result.x
    )


# ============================================================
# AJUSTE MIXTO
# λ = peso Theory
# 1-λ = peso Context
# ============================================================

def fit_mixed(
    E_ctx,
    E_th,
    Y,
    mask,
):

    def objective(theta):

        logit_lambda, log_T = theta

        lam = (
            1.0
            /
            (
                1.0
                + np.exp(-logit_lambda)
            )
        )

        T = np.exp(log_T)

        E_mix = (
            lam * E_th
            + (1.0 - lam) * E_ctx
        )

        P = energy_softmax(
            E_mix,
            mask,
            T
        )

        return cross_entropy(
            Y,
            P,
            mask
        )

    result = minimize(
        objective,
        x0=np.array([0.0, 0.0]),
        method="L-BFGS-B",
        bounds=[
            (-8, 8),
            (-8, 8),
        ]
    )

    logit_lambda, log_T = result.x

    lam = (
        1.0
        /
        (
            1.0
            + np.exp(-logit_lambda)
        )
    )

    T = np.exp(
        log_T
    )

    return lam, T


# ============================================================
# AJUSTE EXCLUSIVAMENTE EN TRAIN
# ============================================================

tr = splits["train"]

T_ctx = fit_temperature(
    E_ctx[tr],
    Y[tr],
    mask[tr]
)

T_th = fit_temperature(
    E_th[tr],
    Y[tr],
    mask[tr]
)

lambda_mix, T_mix = fit_mixed(
    E_ctx[tr],
    E_th[tr],
    Y[tr],
    mask[tr]
)


print(
    f"T_context = {T_ctx:.6f}"
)

print(
    f"T_theory  = {T_th:.6f}"
)

print(
    f"λ_theory   = {lambda_mix:.6f}"
)

print(
    f"λ_context  = {1-lambda_mix:.6f}"
)

print(
    f"T_mixed    = {T_mix:.6f}"
)


# ============================================================
# MÉTRICAS COMPOSICIONALES
# ============================================================

def smooth_simplex(
    x,
    mask,
    eps=EPS,
):

    z = np.where(
        mask,
        np.maximum(x, eps),
        0.0
    )

    return z / z.sum(
        axis=1,
        keepdims=True
    )


def evaluate(
    Y,
    P,
    mask,
):

    Y_s = smooth_simplex(
        Y,
        mask
    )

    P_s = smooth_simplex(
        P,
        mask
    )


    # --------------------------------------------
    # Cross Entropy
    # --------------------------------------------

    CE = cross_entropy(
        Y_s,
        P_s,
        mask
    )


    # --------------------------------------------
    # KL(Y || P)
    # --------------------------------------------

    KL = np.mean(
        np.sum(
            np.where(
                mask,
                Y_s
                * (
                    np.log(Y_s + EPS)
                    - np.log(P_s + EPS)
                ),
                0.0
            ),
            axis=1
        )
    )


    # --------------------------------------------
    # Jensen-Shannon
    # --------------------------------------------

    M = (
        Y_s + P_s
    ) / 2.0

    JS = 0.5 * np.mean(
        np.sum(
            np.where(
                mask,
                Y_s
                * (
                    np.log(Y_s + EPS)
                    - np.log(M + EPS)
                ),
                0.0
            ),
            axis=1
        )
    ) + 0.5 * np.mean(
        np.sum(
            np.where(
                mask,
                P_s
                * (
                    np.log(P_s + EPS)
                    - np.log(M + EPS)
                ),
                0.0
            ),
            axis=1
        )
    )


    # --------------------------------------------
    # RMSE / MAE
    # --------------------------------------------

    diff = np.where(
        mask,
        P_s - Y_s,
        np.nan
    )

    RMSE = np.sqrt(
        np.nanmean(
            diff ** 2
        )
    )

    MAE = np.nanmean(
        np.abs(diff)
    )


    # --------------------------------------------
    # MAE normalizado por activo
    # MAE_j / mean(yp_j)
    # --------------------------------------------

    mae_asset = np.nanmean(
        np.abs(diff),
        axis=0
    )

    mean_y_asset = np.nanmean(
        np.where(
            mask,
            Y_s,
            np.nan
        ),
        axis=0
    )

    nmae_asset = np.divide(
        mae_asset,
        mean_y_asset,
        out=np.full_like(
            mae_asset,
            np.nan
        ),
        where=mean_y_asset > EPS
    )

    NMAE_mean = np.nanmean(
        nmae_asset
    )

    NMAE_median = np.nanmedian(
        nmae_asset
    )


    # --------------------------------------------
    # Aitchison
    # --------------------------------------------

    logY = np.where(
        mask,
        np.log(Y_s + EPS),
        np.nan
    )

    logP = np.where(
        mask,
        np.log(P_s + EPS),
        np.nan
    )

    clrY = (
        logY
        - np.nanmean(
            logY,
            axis=1,
            keepdims=True
        )
    )

    clrP = (
        logP
        - np.nanmean(
            logP,
            axis=1,
            keepdims=True
        )
    )

    Aitchison = np.nanmean(
        np.sqrt(
            np.nansum(
                (clrY - clrP) ** 2,
                axis=1
            )
        )
    )


    return {
        "CE": CE,
        "KL": KL,
        "JS": JS,
        "Aitchison": Aitchison,
        "RMSE": RMSE,
        "MAE": MAE,
        "NMAE_mean": NMAE_mean,
        "NMAE_median": NMAE_median,
    }


# ============================================================
# PREDICCIONES
# ============================================================

P_ctx = energy_softmax(
    E_ctx,
    mask,
    T_ctx
)

P_th = energy_softmax(
    E_th,
    mask,
    T_th
)


E_mix = (
    lambda_mix * E_th
    + (1.0 - lambda_mix) * E_ctx
)

P_mix = energy_softmax(
    E_mix,
    mask,
    T_mix
)


# ============================================================
# BASELINE PERSISTENCIA
# yp_t → yp_t+1
# ============================================================

P_persistence = closure(
    yp_t_valid,
    mask
)


# ============================================================
# EVALUACIÓN
# ============================================================

results = []

models = {
    "Persistence": P_persistence,
    "Softmax_Context": P_ctx,
    "Softmax_Theory": P_th,
    "Softmax_Mixed": P_mix,
}


for split_name in (
    "train",
    "val",
    "test",
):

    s = splits[
        split_name
    ]

    for model_name, P in models.items():

        metrics = evaluate(
            Y[s],
            P[s],
            mask[s]
        )

        results.append({
            "split": split_name,
            "model": model_name,
            **metrics
        })


results = pd.DataFrame(
    results
)

results

Train: 30,540 | Val: 6,544 | Test: 6,545
T_context = 2980.942642
T_theory  = 28.525891
λ_theory   = 0.999665
λ_context  = 0.000335
T_mixed    = 2980.957987


/tmp/ipykernel_2296/3153796897.py:502: RuntimeWarning: Mean of empty slice
  mae_asset = np.nanmean(
/tmp/ipykernel_2296/3153796897.py:507: RuntimeWarning: Mean of empty slice
  mean_y_asset = np.nanmean(


,split,model,CE,KL,JS,Aitchison,RMSE,MAE,NMAE_mean,NMAE_median
0,train,Persistence,3.944655,0.253738,0.055644,20.798139,0.007611,0.001266,0.692213,0.688138
1,train,Softmax_Context,6.902286,3.211369,0.392734,97.573582,0.013554,0.003835,7.749645,5.100496
2,train,Softmax_Theory,5.953839,2.262922,0.383517,38.648083,0.013538,0.003793,7.646274,5.054929
3,train,Softmax_Mixed,6.658662,2.967745,0.390189,85.032205,0.013550,0.003823,7.725332,5.074252
4,val,Persistence,3.676476,0.281839,0.061089,25.309123,0.008850,0.001297,0.793601,0.790216
5,val,Softmax_Context,7.143538,3.748900,0.426189,100.167427,0.015754,0.003938,12.108326,7.267413
6,val,Softmax_Theory,5.972702,2.578064,0.418453,42.033889,0.015736,0.003908,12.008210,7.325704
7,val,Softmax_Mixed,6.809633,3.414995,0.423579,84.849535,0.015749,0.003927,12.084756,7.276558
8,test,Persistence,3.773206,0.360106,0.076296,31.950346,0.010337,0.001462,0.858539,0.860531
9,test,Softmax_Context,7.004288,3.591188,0.421446,105.133259,0.015855,0.003898,10.501437,5.916750


A1 — Resultado preliminar. La parametrización energética basada en N_theory presenta una correspondencia sustancialmente mayor con la geometría composicional observada que la construcción contextual. En test, Softmax_Theory reduce el KL aproximadamente 28.6% y la distancia de Aitchison aproximadamente 56.2% frente a Softmax_Context. Sin embargo, ninguna parametrización energética supera todavía al benchmark de persistencia yp_t → yp_{t+1}, indicando que la correspondencia geométrica no se traduce directamente en superioridad predictiva a horizonte de cinco minutos. El modelo mixto inicial presenta una solución numéricamente subóptima y debe ser reestimado antes de interpretar su parámetro de mezcla.

###A2 Predictiva 2 (mixed robusto)

In [18]:
import numpy as np
import pandas as pd

from scipy.optimize import minimize_scalar, minimize


# ============================================================
# CONFIGURACIÓN EXPERIMENTO A
# ============================================================

EPS = 1e-12

TRAIN_FRAC = 0.70
VAL_FRAC = 0.15

CTX_FEATURE = "E_context"
TH_FEATURE = "E_theory"


# ============================================================
# CARGA
# ============================================================

E_ctx = load_feature(
    df1_ctx,
    assets,
    CTX_FEATURE
).to_numpy(dtype=np.float64, copy=False)

E_th = load_feature(
    df1_th,
    assets,
    TH_FEATURE
).to_numpy(dtype=np.float64, copy=False)

yp_t = load_base(
    df1_ctx,
    assets,
    "yp",
    cfg
).to_numpy(dtype=np.float64, copy=False)

A_t = load_base(
    df1_ctx,
    assets,
    "A",
    cfg
).to_numpy(dtype=np.float64, copy=False)


# ============================================================
# TARGET t+1
# ============================================================

Y = np.empty_like(yp_t)

Y[:-1] = yp_t[1:]
Y[-1] = np.nan


# ============================================================
# SOPORTE COMÚN CAUSAL
# Solo activos conocidos/activos en t y con ambas energías válidas
# ============================================================

mask = (
    (A_t > 0)
    & np.isfinite(E_ctx)
    & np.isfinite(E_th)
    & np.isfinite(Y)
)


# ============================================================
# CIERRE AL SIMPLEX
# ============================================================

def closure(x, mask, eps=EPS):

    z = np.where(
        mask,
        np.maximum(x, 0.0),
        0.0
    )

    total = z.sum(
        axis=1,
        keepdims=True
    )

    return np.divide(
        z,
        total,
        out=np.zeros_like(z),
        where=total > eps
    )


Y = closure(
    Y,
    mask
)


# ============================================================
# FILAS VÁLIDAS
# ============================================================

valid_rows = (
    mask.sum(axis=1) >= 2
) & (
    Y.sum(axis=1) > 1 - 1e-8
)

E_ctx = E_ctx[valid_rows]
E_th = E_th[valid_rows]
yp_t_valid = yp_t[valid_rows]
Y = Y[valid_rows]
mask = mask[valid_rows]


# ============================================================
# SPLIT TEMPORAL
# ============================================================

n = len(Y)

i_train = int(
    n * TRAIN_FRAC
)

i_val = int(
    n * (TRAIN_FRAC + VAL_FRAC)
)

splits = {
    "train": slice(0, i_train),
    "val": slice(i_train, i_val),
    "test": slice(i_val, n),
}

print(
    f"Train: {i_train:,} | "
    f"Val: {i_val - i_train:,} | "
    f"Test: {n - i_val:,}"
)


# ============================================================
# SOFTMAX ENERGÉTICO ESTABLE
# ============================================================

def energy_softmax(
    E,
    mask,
    T,
):

    logits = -E / T

    logits = np.where(
        mask,
        logits,
        -np.inf
    )

    row_max = np.max(
        logits,
        axis=1,
        keepdims=True
    )

    exp_logits = np.where(
        mask,
        np.exp(logits - row_max),
        0.0
    )

    denominator = exp_logits.sum(
        axis=1,
        keepdims=True
    )

    return np.divide(
        exp_logits,
        denominator,
        out=np.zeros_like(exp_logits),
        where=denominator > 0
    )


# ============================================================
# CROSS ENTROPY
# ============================================================

def cross_entropy(
    Y,
    P,
    mask,
    eps=EPS,
):

    return -np.sum(
        np.where(
            mask,
            Y * np.log(P + eps),
            0.0
        )
    ) / len(Y)


# ============================================================
# AJUSTE TEMPERATURA
# ============================================================

def fit_temperature(
    E,
    Y,
    mask,
):

    def objective(log_T):

        T = np.exp(log_T)

        P = energy_softmax(
            E,
            mask,
            T
        )

        return cross_entropy(
            Y,
            P,
            mask
        )

    result = minimize_scalar(
        objective,
        bounds=(-8, 8),
        method="bounded"
    )

    return np.exp(
        result.x
    )


# ============================================================
# MIXED ROBUSTO
# ============================================================

def fit_mixed(
    E_ctx,
    E_th,
    Y,
    mask,
    T_ctx,
    T_th,
):

    def objective(theta):

        beta_th = np.exp(
            theta[0]
        )

        beta_ctx = np.exp(
            theta[1]
        )

        logits = (
            -beta_th * E_th
            -beta_ctx * E_ctx
        )

        logits = np.where(
            mask,
            logits,
            -np.inf
        )

        row_max = np.max(
            logits,
            axis=1,
            keepdims=True
        )

        exp_logits = np.where(
            mask,
            np.exp(
                logits - row_max
            ),
            0.0
        )

        P = exp_logits / exp_logits.sum(
            axis=1,
            keepdims=True
        )

        return cross_entropy(
            Y,
            P,
            mask
        )


    starts = [
        np.log([
            1 / T_th,
            1 / T_ctx,
        ]),
        np.log([
            1 / T_th,
            1e-8,
        ]),
        np.log([
            1e-8,
            1 / T_ctx,
        ]),
        np.log([
            1e-3,
            1e-3,
        ]),
    ]


    results = [
        minimize(
            objective,
            x0=start,
            method="L-BFGS-B",
            bounds=[
                (-20, 10),
                (-20, 10),
            ],
        )
        for start in starts
    ]


    best = min(
        results,
        key=lambda x: x.fun
    )


    beta_th = np.exp(
        best.x[0]
    )

    beta_ctx = np.exp(
        best.x[1]
    )


    beta_sum = (
        beta_th
        + beta_ctx
    )


    lam = (
        beta_th
        / beta_sum
    )

    T = (
        1.0
        / beta_sum
    )


    return (
        beta_th,
        beta_ctx,
        lam,
        T,
    )


# ============================================================
# AJUSTE EXCLUSIVAMENTE EN TRAIN
# ============================================================

tr = splits["train"]

T_ctx = fit_temperature(
    E_ctx[tr],
    Y[tr],
    mask[tr]
)

T_th = fit_temperature(
    E_th[tr],
    Y[tr],
    mask[tr]
)

beta_th, beta_ctx, lambda_mix, T_mix = fit_mixed(
    E_ctx[tr],
    E_th[tr],
    Y[tr],
    mask[tr],
    T_ctx,
    T_th,
)

print(
    f"T_context = {T_ctx:.6f}"
)

print(
    f"T_theory  = {T_th:.6f}"
)

print(
    "β Theory:",
    beta_th
)

print(
    "β Context:",
    beta_ctx
)

print(
    "λ Theory:",
    lambda_mix
)

print(
    "λ Context:",
    1 - lambda_mix
)

print(
    "T Mixed:",
    T_mix
)

# ============================================================
# MÉTRICAS COMPOSICIONALES
# ============================================================

def smooth_simplex(
    x,
    mask,
    eps=EPS,
):

    z = np.where(
        mask,
        np.maximum(x, eps),
        0.0
    )

    return z / z.sum(
        axis=1,
        keepdims=True
    )


def evaluate(
    Y,
    P,
    mask,
):

    Y_s = smooth_simplex(
        Y,
        mask
    )

    P_s = smooth_simplex(
        P,
        mask
    )


    # --------------------------------------------
    # Cross Entropy
    # --------------------------------------------

    CE = cross_entropy(
        Y_s,
        P_s,
        mask
    )


    # --------------------------------------------
    # KL(Y || P)
    # --------------------------------------------

    KL = np.mean(
        np.sum(
            np.where(
                mask,
                Y_s
                * (
                    np.log(Y_s + EPS)
                    - np.log(P_s + EPS)
                ),
                0.0
            ),
            axis=1
        )
    )


    # --------------------------------------------
    # Jensen-Shannon
    # --------------------------------------------

    M = (
        Y_s + P_s
    ) / 2.0

    JS = 0.5 * np.mean(
        np.sum(
            np.where(
                mask,
                Y_s
                * (
                    np.log(Y_s + EPS)
                    - np.log(M + EPS)
                ),
                0.0
            ),
            axis=1
        )
    ) + 0.5 * np.mean(
        np.sum(
            np.where(
                mask,
                P_s
                * (
                    np.log(P_s + EPS)
                    - np.log(M + EPS)
                ),
                0.0
            ),
            axis=1
        )
    )


    # --------------------------------------------
    # RMSE / MAE
    # --------------------------------------------

    diff = np.where(
        mask,
        P_s - Y_s,
        np.nan
    )

    RMSE = np.sqrt(
        np.nanmean(
            diff ** 2
        )
    )

    MAE = np.nanmean(
        np.abs(diff)
    )


    # --------------------------------------------
    # MAE normalizado por activo
    # MAE_j / mean(yp_j)
    # --------------------------------------------

    mae_asset = np.nanmean(
        np.abs(diff),
        axis=0
    )

    mean_y_asset = np.nanmean(
        np.where(
            mask,
            Y_s,
            np.nan
        ),
        axis=0
    )

    nmae_asset = np.divide(
        mae_asset,
        mean_y_asset,
        out=np.full_like(
            mae_asset,
            np.nan
        ),
        where=mean_y_asset > EPS
    )

    NMAE_mean = np.nanmean(
        nmae_asset
    )

    NMAE_median = np.nanmedian(
        nmae_asset
    )


    # --------------------------------------------
    # Aitchison
    # --------------------------------------------

    logY = np.where(
        mask,
        np.log(Y_s + EPS),
        np.nan
    )

    logP = np.where(
        mask,
        np.log(P_s + EPS),
        np.nan
    )

    clrY = (
        logY
        - np.nanmean(
            logY,
            axis=1,
            keepdims=True
        )
    )

    clrP = (
        logP
        - np.nanmean(
            logP,
            axis=1,
            keepdims=True
        )
    )

    Aitchison = np.nanmean(
        np.sqrt(
            np.nansum(
                (clrY - clrP) ** 2,
                axis=1
            )
        )
    )


    return {
        "CE": CE,
        "KL": KL,
        "JS": JS,
        "Aitchison": Aitchison,
        "RMSE": RMSE,
        "MAE": MAE,
        "NMAE_mean": NMAE_mean,
        "NMAE_median": NMAE_median,
    }


# ============================================================
# PREDICCIONES
# ============================================================

P_ctx = energy_softmax(
    E_ctx,
    mask,
    T_ctx
)

P_th = energy_softmax(
    E_th,
    mask,
    T_th
)


# ============================================================
# PREDICCIÓN MIXED
# ============================================================

logits_mix = (
    -beta_th * E_th
    -beta_ctx * E_ctx
)

logits_mix = np.where(
    mask,
    logits_mix,
    -np.inf
)

row_max = np.max(
    logits_mix,
    axis=1,
    keepdims=True
)

exp_logits = np.where(
    mask,
    np.exp(
        logits_mix - row_max
    ),
    0.0
)

P_mix = exp_logits / exp_logits.sum(
    axis=1,
    keepdims=True
)


# ============================================================
# BASELINE PERSISTENCIA
# yp_t → yp_t+1
# ============================================================

P_persistence = closure(
    yp_t_valid,
    mask
)


# ============================================================
# EVALUACIÓN
# ============================================================

results = []

models = {
    "Persistence": P_persistence,
    "Softmax_Context": P_ctx,
    "Softmax_Theory": P_th,
    "Softmax_Mixed": P_mix,
}


for split_name in (
    "train",
    "val",
    "test",
):

    s = splits[
        split_name
    ]

    for model_name, P in models.items():

        metrics = evaluate(
            Y[s],
            P[s],
            mask[s]
        )

        results.append({
            "split": split_name,
            "model": model_name,
            **metrics
        })


results = pd.DataFrame(
    results
)

results

Train: 30,540 | Val: 6,544 | Test: 6,545
T_context = 2980.942642
T_theory  = 28.525891
β Theory: 0.035485899268842075
β Context: 2.061153622438558e-09
λ Theory: 0.9999999419162668
λ Context: 5.8083733156522044e-08
T Mixed: 28.18020573017586


/tmp/ipykernel_2296/4265399291.py:576: RuntimeWarning: Mean of empty slice
  mae_asset = np.nanmean(
/tmp/ipykernel_2296/4265399291.py:581: RuntimeWarning: Mean of empty slice
  mean_y_asset = np.nanmean(


,split,model,CE,KL,JS,Aitchison,RMSE,MAE,NMAE_mean,NMAE_median
0,train,Persistence,3.944655,0.253738,0.055644,20.798139,0.007611,0.001266,0.692213,0.688138
1,train,Softmax_Context,6.902286,3.211369,0.392734,97.573582,0.013554,0.003835,7.749645,5.100496
2,train,Softmax_Theory,5.953839,2.262922,0.383517,38.648083,0.013538,0.003793,7.646274,5.054929
3,train,Softmax_Mixed,6.592359,2.901442,0.389342,82.265705,0.013546,0.003820,7.710959,5.100165
4,val,Persistence,3.676476,0.281839,0.061089,25.309123,0.008850,0.001297,0.793601,0.790216
5,val,Softmax_Context,7.143538,3.748900,0.426189,100.167427,0.015754,0.003938,12.108326,7.267413
6,val,Softmax_Theory,5.972702,2.578064,0.418453,42.033889,0.015736,0.003908,12.008210,7.325704
7,val,Softmax_Mixed,6.761353,3.366716,0.424038,79.680356,0.015747,0.003932,12.105995,7.336295
8,test,Persistence,3.773206,0.360106,0.076296,31.950346,0.010337,0.001462,0.858539,0.860531
9,test,Softmax_Context,7.004288,3.591188,0.421446,105.133259,0.015855,0.003898,10.501437,5.916750


la tabla actual tiene una inconsistencia: Softmax_Mixed debería producir métricas casi idénticas —o ligeramente mejores en Train— que Softmax_Theory

In [19]:
# ============================================================
# DIAGNÓSTICO DEL MODELO MIXTO
# ============================================================

beta_theory_pure = 1.0 / T_th

print(f"β Theory puro : {beta_theory_pure:.12f}")
print(f"β Theory mixed: {beta_th:.12f}")
print(f"β Context mix : {beta_ctx:.12e}")


# ============================================================
# RECONSTRUCCIÓN DIRECTA DE P_mix
# ============================================================

logits_mix_check = (
    -beta_th * E_th
    -beta_ctx * E_ctx
)

logits_mix_check = np.where(
    mask,
    logits_mix_check,
    -np.inf
)

row_max = np.max(
    logits_mix_check,
    axis=1,
    keepdims=True
)

exp_logits = np.where(
    mask,
    np.exp(logits_mix_check - row_max),
    0.0
)

P_mix_check = np.divide(
    exp_logits,
    exp_logits.sum(axis=1, keepdims=True),
    out=np.zeros_like(exp_logits),
    where=exp_logits.sum(axis=1, keepdims=True) > 0
)


# ============================================================
# COMPARACIÓN
# ============================================================

print(
    "\nCE Theory Train:",
    cross_entropy(
        Y[tr],
        P_th[tr],
        mask[tr]
    )
)

print(
    "CE Mixed Train:",
    cross_entropy(
        Y[tr],
        P_mix_check[tr],
        mask[tr]
    )
)

print(
    "\nMáxima diferencia Mixed vs Theory:",
    np.nanmax(
        np.abs(
            P_mix_check - P_th
        )
    )
)

print(
    "Máxima diferencia P_mix actual vs reconstruido:",
    np.nanmax(
        np.abs(
            P_mix - P_mix_check
        )
    )
)

β Theory puro : 0.035055872292
β Theory mixed: 0.035485899269
β Context mix : 2.061153622439e-09

CE Theory Train: 5.953839207655138
CE Mixed Train: 6.6143030331394606

Máxima diferencia Mixed vs Theory: 0.003307060753763452
Máxima diferencia P_mix actual vs reconstruido: 0.0


In [20]:
# ============================================================
# PREDICCIÓN MIXTA FINAL
# ============================================================

logits_mix = (
    -beta_th * E_th
    -beta_ctx * E_ctx
)

logits_mix = np.where(
    mask,
    logits_mix,
    -np.inf
)

row_max = np.max(
    logits_mix,
    axis=1,
    keepdims=True
)

exp_logits = np.where(
    mask,
    np.exp(logits_mix - row_max),
    0.0
)

denominator = exp_logits.sum(
    axis=1,
    keepdims=True
)

P_mix = np.divide(
    exp_logits,
    denominator,
    out=np.zeros_like(exp_logits),
    where=denominator > 0
)

In [21]:
# ============================================================
# EVALUACIÓN FINAL
# ============================================================

results = []

models = {
    "Persistence": P_persistence,
    "Softmax_Context": P_ctx,
    "Softmax_Theory": P_th,
    "Softmax_Mixed": P_mix,
}

for split_name in (
    "train",
    "val",
    "test",
):

    s = splits[
        split_name
    ]

    for model_name, P in models.items():

        metrics = evaluate(
            Y[s],
            P[s],
            mask[s]
        )

        results.append({
            "split": split_name,
            "model": model_name,
            **metrics
        })


results = pd.DataFrame(
    results
)

results

/tmp/ipykernel_2296/4265399291.py:576: RuntimeWarning: Mean of empty slice
  mae_asset = np.nanmean(
/tmp/ipykernel_2296/4265399291.py:581: RuntimeWarning: Mean of empty slice
  mean_y_asset = np.nanmean(


,split,model,CE,KL,JS,Aitchison,RMSE,MAE,NMAE_mean,NMAE_median
0,train,Persistence,3.944655,0.253738,0.055644,20.798139,0.007611,0.001266,0.692213,0.688138
1,train,Softmax_Context,6.902286,3.211369,0.392734,97.573582,0.013554,0.003835,7.749645,5.100496
2,train,Softmax_Theory,5.953839,2.262922,0.383517,38.648083,0.013538,0.003793,7.646274,5.054929
3,train,Softmax_Mixed,6.592359,2.901442,0.389342,82.265705,0.013546,0.003820,7.710959,5.100165
4,val,Persistence,3.676476,0.281839,0.061089,25.309123,0.008850,0.001297,0.793601,0.790216
5,val,Softmax_Context,7.143538,3.748900,0.426189,100.167427,0.015754,0.003938,12.108326,7.267413
6,val,Softmax_Theory,5.972702,2.578064,0.418453,42.033889,0.015736,0.003908,12.008210,7.325704
7,val,Softmax_Mixed,6.761353,3.366716,0.424038,79.680356,0.015747,0.003932,12.105995,7.336295
8,test,Persistence,3.773206,0.360106,0.076296,31.950346,0.010337,0.001462,0.858539,0.860531
9,test,Softmax_Context,7.004288,3.591188,0.421446,105.133259,0.015855,0.003898,10.501437,5.916750


Hasta este punto, la evidencia apunta a cuatro resultados diferentes.

- Primero, la formulación teórica de energía posee una relación mucho más fuerte con la estructura composicional de yp que la formulación contextual. Esto aparece especialmente en KL y Aitchison.

- Segundo, esta ventaja no es marginal. En geometría Aitchison, Theory reduce más de la mitad de la distancia observada respecto de Context.

- Tercero, la estructura energética por sí sola todavía no sustituye la enorme persistencia temporal del simplex a cinco minutos. El benchmark yp_t sigue siendo claramente superior para predecir yp_{t+1}.

- Cuarto, cuando permitimos competir simultáneamente a ambas energías, el ajuste lleva prácticamente todo el peso hacia Theory: beta_{ctx} tienda a 0

Esto es compatible con la hipótesis de que la información útil capturada por Context queda ampliamente contenida —o dominada— por la geometría de Theory dentro de esta parametrización. Pero el problema numérico del Mixed impide todavía afirmar que el modelo conjunto confirma formalmente esa dominancia.

La representación energética derivada de la rama teórica presenta una correspondencia sustancialmente mayor con la geometría del simplex económico observado que la representación energética puramente contextual, aunque esa correspondencia todavía no supera la persistencia del simplex como mecanismo predictivo a cinco minutos.

###A3 Descrptiva

In [22]:
# ============================================================
# EXPERIMENTO A3
# RECONSTRUCCIÓN CONTEMPORÁNEA DEL SIMPLEX
#
# E_context(t) -> yp(t)
# E_theory(t)  -> yp(t)
# E_mixed(t)   -> yp(t)
# ============================================================

import numpy as np
import pandas as pd

from scipy.optimize import minimize_scalar, minimize


# ============================================================
# CONFIGURACIÓN
# ============================================================

EPS = 1e-12

TRAIN_FRAC = 0.70
VAL_FRAC = 0.15

FORBIDDEN_FEATURES = (
    "FLOW_TOTAL_AUDIT",
    "TURNOVER_PER_TRADE_AUDIT",
)


# ============================================================
# CONTROL EXPLÍCITO DE VARIABLES PROHIBIDAS
# ============================================================

selected_columns = (
    [f"{a}__E_context" for a in assets]
    + [f"{a}__E_theory" for a in assets]
    + [f"{a}_yp" for a in assets]
    + [f"{a}{cfg.active_suffix}" for a in assets]
)

for forbidden in FORBIDDEN_FEATURES:
    assert not any(
        forbidden in c
        for c in selected_columns
    ), f"Variable prohibida detectada: {forbidden}"

print("Variables audit excluidas correctamente.")


# ============================================================
# CARGA
# ============================================================

E_ctx = load_feature(
    df1_ctx,
    assets,
    "E_context"
).to_numpy(
    dtype=np.float64,
    copy=False
)

E_th = load_feature(
    df1_th,
    assets,
    "E_theory"
).to_numpy(
    dtype=np.float64,
    copy=False
)

yp_t = load_base(
    df1_ctx,
    assets,
    "yp",
    cfg
).to_numpy(
    dtype=np.float64,
    copy=False
)

A_t = load_base(
    df1_ctx,
    assets,
    "A",
    cfg
).to_numpy(
    dtype=np.float64,
    copy=False
)


# ============================================================
# TARGET CONTEMPORÁNEO
# yp(t), NO yp(t+1)
# ============================================================

Y = yp_t.copy()


# ============================================================
# SOPORTE COMÚN
# ============================================================

mask = (
    (A_t > 0)
    & np.isfinite(E_ctx)
    & np.isfinite(E_th)
    & np.isfinite(Y)
)


# ============================================================
# CIERRE AL SIMPLEX
# ============================================================

def closure(
    x,
    mask,
    eps=EPS
):

    z = np.where(
        mask,
        np.maximum(x, 0.0),
        0.0
    )

    total = z.sum(
        axis=1,
        keepdims=True
    )

    return np.divide(
        z,
        total,
        out=np.zeros_like(z),
        where=total > eps
    )


Y = closure(
    Y,
    mask
)


# ============================================================
# FILAS VÁLIDAS
# ============================================================

valid_rows = (
    (mask.sum(axis=1) >= 2)
    & (Y.sum(axis=1) > 1 - 1e-8)
)

E_ctx = E_ctx[
    valid_rows
]

E_th = E_th[
    valid_rows
]

Y = Y[
    valid_rows
]

mask = mask[
    valid_rows
]


# ============================================================
# SPLIT TEMPORAL
# Calibración -> estabilidad descriptiva futura
# ============================================================

n = len(Y)

i_train = int(
    n * TRAIN_FRAC
)

i_val = int(
    n * (TRAIN_FRAC + VAL_FRAC)
)

splits = {
    "train": slice(
        0,
        i_train
    ),

    "val": slice(
        i_train,
        i_val
    ),

    "test": slice(
        i_val,
        n
    ),
}

print(
    f"Train: {i_train:,} | "
    f"Val: {i_val - i_train:,} | "
    f"Test: {n - i_val:,}"
)


# ============================================================
# SOFTMAX ENERGÉTICO
# ============================================================

def energy_softmax(
    E,
    mask,
    T
):

    logits = np.where(
        mask,
        -E / T,
        -np.inf
    )

    row_max = np.max(
        logits,
        axis=1,
        keepdims=True
    )

    exp_logits = np.where(
        mask,
        np.exp(
            logits - row_max
        ),
        0.0
    )

    denominator = exp_logits.sum(
        axis=1,
        keepdims=True
    )

    return np.divide(
        exp_logits,
        denominator,
        out=np.zeros_like(exp_logits),
        where=denominator > 0
    )


# ============================================================
# SOFTMAX MIXTO
# ============================================================

def mixed_softmax(
    E_th,
    E_ctx,
    mask,
    beta_th,
    beta_ctx
):

    logits = np.where(
        mask,
        (
            -beta_th * E_th
            -beta_ctx * E_ctx
        ),
        -np.inf
    )

    row_max = np.max(
        logits,
        axis=1,
        keepdims=True
    )

    exp_logits = np.where(
        mask,
        np.exp(
            logits - row_max
        ),
        0.0
    )

    denominator = exp_logits.sum(
        axis=1,
        keepdims=True
    )

    return np.divide(
        exp_logits,
        denominator,
        out=np.zeros_like(exp_logits),
        where=denominator > 0
    )


# ============================================================
# CROSS ENTROPY
# ============================================================

def cross_entropy(
    Y,
    P,
    mask,
    eps=EPS
):

    return -np.sum(
        np.where(
            mask,
            Y * np.log(P + eps),
            0.0
        )
    ) / len(Y)


# ============================================================
# AJUSTE TEMPERATURA
# ============================================================

def fit_temperature(
    E,
    Y,
    mask
):

    def objective(
        log_T
    ):

        T = np.exp(
            log_T
        )

        P = energy_softmax(
            E,
            mask,
            T
        )

        return cross_entropy(
            Y,
            P,
            mask
        )

    result = minimize_scalar(
        objective,
        bounds=(-8, 12),
        method="bounded"
    )

    return np.exp(
        result.x
    )


# ============================================================
# AJUSTE MIXTO
# beta >= 0
# Incluye los modelos puros como puntos iniciales
# ============================================================

def fit_mixed(
    E_ctx,
    E_th,
    Y,
    mask,
    T_ctx,
    T_th
):

    def objective(
        beta
    ):

        P = mixed_softmax(
            E_th,
            E_ctx,
            mask,
            beta[0],
            beta[1]
        )

        return cross_entropy(
            Y,
            P,
            mask
        )


    starts = [
        [1.0 / T_th, 0.0],
        [0.0, 1.0 / T_ctx],
        [1.0 / T_th, 1.0 / T_ctx],
        [0.001, 0.001],
    ]


    candidates = []


    for start in starts:

        result = minimize(
            objective,
            x0=np.asarray(
                start,
                dtype=np.float64
            ),
            method="L-BFGS-B",
            bounds=[
                (0.0, None),
                (0.0, None),
            ],
        )

        candidates.append(
            result
        )


    best = min(
        candidates,
        key=lambda x: x.fun
    )

    beta_th = best.x[0]
    beta_ctx = best.x[1]

    return (
        beta_th,
        beta_ctx,
        best.fun
    )


# ============================================================
# CALIBRACIÓN SOLO EN TRAIN
# ============================================================

tr = splits[
    "train"
]


T_ctx = fit_temperature(
    E_ctx[tr],
    Y[tr],
    mask[tr]
)

T_th = fit_temperature(
    E_th[tr],
    Y[tr],
    mask[tr]
)


beta_th, beta_ctx, CE_mix_train = fit_mixed(
    E_ctx[tr],
    E_th[tr],
    Y[tr],
    mask[tr],
    T_ctx,
    T_th
)


beta_sum = (
    beta_th
    + beta_ctx
)

lambda_th = (
    beta_th / beta_sum
    if beta_sum > 0
    else np.nan
)

lambda_ctx = (
    beta_ctx / beta_sum
    if beta_sum > 0
    else np.nan
)

T_mix = (
    1.0 / beta_sum
    if beta_sum > 0
    else np.inf
)


print(
    f"T_context = {T_ctx:.8f}"
)

print(
    f"T_theory  = {T_th:.8f}"
)

print(
    f"β Theory  = {beta_th:.12f}"
)

print(
    f"β Context = {beta_ctx:.12f}"
)

print(
    f"λ Theory  = {lambda_th:.8f}"
)

print(
    f"λ Context = {lambda_ctx:.8f}"
)

print(
    f"T Mixed   = {T_mix:.8f}"
)


# ============================================================
# PREDICCIONES CONTEMPORÁNEAS
# ============================================================

P_ctx = energy_softmax(
    E_ctx,
    mask,
    T_ctx
)

P_th = energy_softmax(
    E_th,
    mask,
    T_th
)

P_mix = mixed_softmax(
    E_th,
    E_ctx,
    mask,
    beta_th,
    beta_ctx
)


# ============================================================
# CONTROL DE SANIDAD DEL MIXED
# ============================================================

ce_ctx_train = cross_entropy(
    Y[tr],
    P_ctx[tr],
    mask[tr]
)

ce_th_train = cross_entropy(
    Y[tr],
    P_th[tr],
    mask[tr]
)

ce_mix_train = cross_entropy(
    Y[tr],
    P_mix[tr],
    mask[tr]
)

print(
    "\nCE Train Context:",
    ce_ctx_train
)

print(
    "CE Train Theory :",
    ce_th_train
)

print(
    "CE Train Mixed  :",
    ce_mix_train
)

assert (
    ce_mix_train
    <= min(
        ce_ctx_train,
        ce_th_train
    ) + 1e-7
), "Mixed no alcanzó al mejor modelo puro."


# ============================================================
# SUAVIZADO COMPOSICIONAL
# ============================================================

def smooth_simplex(
    x,
    mask,
    eps=EPS
):

    z = np.where(
        mask,
        np.maximum(
            x,
            eps
        ),
        0.0
    )

    total = z.sum(
        axis=1,
        keepdims=True
    )

    return np.divide(
        z,
        total,
        out=np.zeros_like(z),
        where=total > 0
    )


# ============================================================
# MÉTRICAS
# ============================================================

def evaluate(
    Y,
    P,
    mask
):

    Y_s = smooth_simplex(
        Y,
        mask
    )

    P_s = smooth_simplex(
        P,
        mask
    )


    # --------------------------------------------
    # Cross Entropy
    # --------------------------------------------

    CE = cross_entropy(
        Y_s,
        P_s,
        mask
    )


    # --------------------------------------------
    # KL
    # --------------------------------------------

    KL = np.mean(
        np.sum(
            np.where(
                mask,
                Y_s * (
                    np.log(Y_s + EPS)
                    - np.log(P_s + EPS)
                ),
                0.0
            ),
            axis=1
        )
    )


    # --------------------------------------------
    # Jensen-Shannon
    # --------------------------------------------

    M = (
        Y_s + P_s
    ) / 2.0

    KL_YM = np.sum(
        np.where(
            mask,
            Y_s * (
                np.log(Y_s + EPS)
                - np.log(M + EPS)
            ),
            0.0
        ),
        axis=1
    )

    KL_PM = np.sum(
        np.where(
            mask,
            P_s * (
                np.log(P_s + EPS)
                - np.log(M + EPS)
            ),
            0.0
        ),
        axis=1
    )

    JS = np.mean(
        0.5 * KL_YM
        + 0.5 * KL_PM
    )


    # --------------------------------------------
    # RMSE / MAE
    # --------------------------------------------

    diff = np.where(
        mask,
        P_s - Y_s,
        np.nan
    )

    RMSE = np.sqrt(
        np.nanmean(
            diff ** 2
        )
    )

    MAE = np.nanmean(
        np.abs(diff)
    )


    # --------------------------------------------
    # MAE NORMALIZADO POR ACTIVO
    # --------------------------------------------

    mae_asset = np.full(
        Y.shape[1],
        np.nan,
        dtype=np.float64
    )

    mean_y_asset = np.full(
        Y.shape[1],
        np.nan,
        dtype=np.float64
    )


    for j in range(
        Y.shape[1]
    ):

        valid = mask[:, j]

        if valid.any():

            mae_asset[j] = np.mean(
                np.abs(
                    P_s[valid, j]
                    - Y_s[valid, j]
                )
            )

            mean_y_asset[j] = np.mean(
                Y_s[valid, j]
            )


    nmae_asset = np.divide(
        mae_asset,
        mean_y_asset,
        out=np.full_like(
            mae_asset,
            np.nan
        ),
        where=mean_y_asset > EPS
    )

    NMAE_mean = np.nanmean(
        nmae_asset
    )

    NMAE_median = np.nanmedian(
        nmae_asset
    )


    # --------------------------------------------
    # Aitchison
    # --------------------------------------------

    logY = np.where(
        mask,
        np.log(Y_s + EPS),
        np.nan
    )

    logP = np.where(
        mask,
        np.log(P_s + EPS),
        np.nan
    )


    clrY = (
        logY
        - np.nanmean(
            logY,
            axis=1,
            keepdims=True
        )
    )

    clrP = (
        logP
        - np.nanmean(
            logP,
            axis=1,
            keepdims=True
        )
    )


    Aitchison = np.nanmean(
        np.sqrt(
            np.nansum(
                (
                    clrY - clrP
                ) ** 2,
                axis=1
            )
        )
    )


    return {
        "CE": CE,
        "KL": KL,
        "JS": JS,
        "Aitchison": Aitchison,
        "RMSE": RMSE,
        "MAE": MAE,
        "NMAE_mean": NMAE_mean,
        "NMAE_median": NMAE_median,
    }


# ============================================================
# EVALUACIÓN DESCRIPTIVA
# ============================================================

results_desc = []

models = {
    "Softmax_Context": P_ctx,
    "Softmax_Theory": P_th,
    "Softmax_Mixed": P_mix,
}


for split_name in (
    "train",
    "val",
    "test",
):

    s = splits[
        split_name
    ]

    for model_name, P in models.items():

        metrics = evaluate(
            Y[s],
            P[s],
            mask[s]
        )

        results_desc.append({
            "split": split_name,
            "model": model_name,
            **metrics
        })


results_desc = pd.DataFrame(
    results_desc
)

results_desc

Variables audit excluidas correctamente.
Train: 30,540 | Val: 6,545 | Test: 6,545
T_context = 162753.71012078
T_theory  = 23.83747719
β Theory  = 0.041950748060
β Context = 0.000000000000
λ Theory  = 1.00000000
λ Context = 0.00000000
T Mixed   = 23.83747719

CE Train Context: 6.777937533610021
CE Train Theory : 5.950933984789362
CE Train Mixed  : 5.950933984789362


,split,model,CE,KL,JS,Aitchison,RMSE,MAE,NMAE_mean,NMAE_median
0,train,Softmax_Context,6.751744,3.060759,0.391280,90.082811,0.013551,0.003828,7.709208,5.088893
1,train,Softmax_Theory,5.950934,2.259949,0.383350,37.599496,0.013537,0.003792,7.616863,5.070309
2,train,Softmax_Mixed,5.950934,2.259949,0.383350,37.599496,0.013537,0.003792,7.616863,5.070309
3,val,Softmax_Context,6.920822,3.525883,0.424433,91.606044,0.015750,0.003931,12.019336,7.258995
4,val,Softmax_Theory,5.969848,2.574908,0.418242,39.308739,0.015734,0.003908,11.934153,7.292184
5,val,Softmax_Mixed,5.969848,2.574908,0.418242,39.308739,0.015734,0.003908,11.934153,7.292184
6,test,Softmax_Context,6.827738,3.413822,0.419189,95.992512,0.015850,0.003889,10.398746,5.826256
7,test,Softmax_Theory,5.973685,2.559769,0.414280,40.107158,0.015840,0.003870,10.497914,5.922281
8,test,Softmax_Mixed,5.973685,2.559769,0.414280,40.107158,0.015840,0.003870,10.497914,5.922281


A3 — Reconstrucción contemporánea. Al evaluar la capacidad de las representaciones energéticas para reconstruir el simplex económico contemporáneo yp_t, la energía teórica presenta una correspondencia composicional sustancialmente superior a la energía contextual. En Test, E_theory reduce aproximadamente 25% la divergencia KL y 58% la distancia de Aitchison respecto de E_context.

Al permitir una combinación libre de ambas energías, el óptimo asigna peso nulo a E_context y reproduce exactamente el modelo teórico, indicando que dentro de esta parametrización softmax la energía contextual no aporta información descriptiva marginal una vez conocida la energía teórica. Dado que N_theory incorpora yp_t en su construcción, este resultado constituye evidencia de consistencia interna y adecuación geométrica del mecanismo teórico, no una validación predictiva independiente.

### A4 Descriptivo metricas de error y calidad

In [25]:
# ============================================================
# EXPERIMENTO PREDICTIVO - NAMESPACE INDEPENDIENTE
# E(t) -> yp(t+1)
# ============================================================

import numpy as np
import pandas as pd

from scipy.optimize import minimize, minimize_scalar


# ============================================================
# CONFIGURACIÓN
# ============================================================

EPS = 1e-12
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15

FORBIDDEN_FEATURES = (
    "FLOW_TOTAL_AUDIT",
    "TURNOVER_PER_TRADE_AUDIT",
)


# ============================================================
# CONTROL DE VARIABLES PROHIBIDAS
# ============================================================

used_columns = (
    [f"{a}__E_context" for a in assets]
    + [f"{a}__E_theory" for a in assets]
    + [f"{a}_yp" for a in assets]
    + [f"{a}{cfg.active_suffix}" for a in assets]
)

assert not any(
    forbidden in col
    for forbidden in FORBIDDEN_FEATURES
    for col in used_columns
)

print("Variables AUDIT excluidas correctamente.")


# ============================================================
# CARGA
# ============================================================

pred_Ectx = load_feature(
    df1_ctx,
    assets,
    "E_context"
).to_numpy(
    dtype=np.float64,
    copy=False
)

pred_Eth = load_feature(
    df1_th,
    assets,
    "E_theory"
).to_numpy(
    dtype=np.float64,
    copy=False
)

pred_yp_t = load_base(
    df1_ctx,
    assets,
    "yp",
    cfg
).to_numpy(
    dtype=np.float64,
    copy=False
)

pred_A = load_base(
    df1_ctx,
    assets,
    "A",
    cfg
).to_numpy(
    dtype=np.float64,
    copy=False
)


# ============================================================
# TARGET t+1
# ============================================================

pred_Y_raw = np.full_like(
    pred_yp_t,
    np.nan
)

pred_Y_raw[:-1] = pred_yp_t[1:]


# ============================================================
# SOPORTE COMÚN
# ============================================================

pred_mask = (
    (pred_A > 0)
    & np.isfinite(pred_Ectx)
    & np.isfinite(pred_Eth)
    & np.isfinite(pred_Y_raw)
)


# ============================================================
# CIERRE
# ============================================================

def pred_closure(x, mask, eps=EPS):

    z = np.where(
        mask,
        np.maximum(x, 0.0),
        0.0
    )

    total = z.sum(
        axis=1,
        keepdims=True
    )

    return np.divide(
        z,
        total,
        out=np.zeros_like(z),
        where=total > eps
    )


pred_Y = pred_closure(
    pred_Y_raw,
    pred_mask
)


# ============================================================
# FILAS VÁLIDAS
# ============================================================

pred_valid_rows = (
    (pred_mask.sum(axis=1) >= 2)
    & (pred_Y.sum(axis=1) > 1 - 1e-8)
)

pred_Ectx = pred_Ectx[pred_valid_rows]
pred_Eth = pred_Eth[pred_valid_rows]

pred_yp_current = pred_yp_t[pred_valid_rows]
pred_Y_raw = pred_Y_raw[pred_valid_rows]

pred_Y = pred_Y[pred_valid_rows]
pred_mask = pred_mask[pred_valid_rows]


# ============================================================
# SPLIT TEMPORAL
# ============================================================

n = len(pred_Y)

i_train = int(
    n * TRAIN_FRAC
)

i_val = int(
    n * (TRAIN_FRAC + VAL_FRAC)
)

pred_splits = {
    "train": slice(0, i_train),
    "val": slice(i_train, i_val),
    "test": slice(i_val, n),
}

print(
    f"Train: {i_train:,} | "
    f"Val: {i_val-i_train:,} | "
    f"Test: {n-i_val:,}"
)


# ============================================================
# yp_mean EXCLUSIVAMENTE EN TRAIN
# ============================================================

tr = pred_splits["train"]

yp_mean_train = np.nanmean(
    pred_Y_raw[tr],
    axis=0
)

yp_mean = pd.Series(
    yp_mean_train,
    index=assets,
    name="yp_mean_train"
).sort_values(
    ascending=False
)


# ============================================================
# TOP / BOTTOM
# ============================================================

n_assets = len(assets)

n_top1 = int(
    np.ceil(
        n_assets * 0.01
    )
)

n_top5 = int(
    np.ceil(
        n_assets * 0.05
    )
)

n_bottom50 = int(
    np.floor(
        n_assets * 0.50
    )
)


top1_assets = yp_mean.head(
    n_top1
).index.tolist()

top5_assets = yp_mean.head(
    n_top5
).index.tolist()

bottom50_assets = yp_mean.tail(
    n_bottom50
).index.tolist()


print("\nTop 1%:", n_top1)
print("Top 5%:", n_top5)
print("Bottom 50%:", n_bottom50)


# ============================================================
# TABLA DE GRUPOS
# ============================================================

groups_df = pd.concat([
    yp_mean.loc[top1_assets].rename("Top_1%"),
    yp_mean.loc[top5_assets].rename("Top_5%"),
    yp_mean.loc[bottom50_assets].rename("Bottom_50%"),
], axis=1)

print("\nTOP 1%")
display(
    yp_mean.loc[
        top1_assets
    ].to_frame()
)

print("\nTOP 5%")
display(
    yp_mean.loc[
        top5_assets
    ].to_frame()
)

print("\nBOTTOM 50%")
display(
    yp_mean.loc[
        bottom50_assets
    ].to_frame()
)

Variables AUDIT excluidas correctamente.
Train: 30,540 | Val: 6,544 | Test: 6,545

Top 1%: 6
Top 5%: 26
Bottom 50%: 255

TOP 1%


,yp_mean_train
BTC,0.147572
ETH,0.086714
USDC,0.066578
XRP,0.055162
SOL,0.052947
DOGE,0.045686



TOP 5%


,yp_mean_train
BTC,0.147572
ETH,0.086714
USDC,0.066578
XRP,0.055162
SOL,0.052947
DOGE,0.045686
FDUSD,0.039968
PEPE,0.025130
SUI,0.018685
BNB,0.017233



BOTTOM 50%


,yp_mean_train
ENJ,0.000317
JST,0.000316
METIS,0.000311
POWR,0.000310
SNT,0.000308
...,...
A,0.000000
ZKC,0.000000
ZKP,0.000000
0G,0.000000


In [26]:
# ============================================================
# SOFTMAX ENERGÉTICO
# ============================================================

def pred_energy_softmax(
    E,
    mask,
    T
):

    logits = np.where(
        mask,
        -E / T,
        -np.inf
    )

    row_max = np.max(
        logits,
        axis=1,
        keepdims=True
    )

    exp_logits = np.where(
        mask,
        np.exp(
            logits - row_max
        ),
        0.0
    )

    denominator = exp_logits.sum(
        axis=1,
        keepdims=True
    )

    return np.divide(
        exp_logits,
        denominator,
        out=np.zeros_like(exp_logits),
        where=denominator > 0
    )


# ============================================================
# MIXED SOFTMAX
# ============================================================

def pred_mixed_softmax(
    E_th,
    E_ctx,
    mask,
    beta_th,
    beta_ctx
):

    logits = np.where(
        mask,
        (
            -beta_th * E_th
            -beta_ctx * E_ctx
        ),
        -np.inf
    )

    row_max = np.max(
        logits,
        axis=1,
        keepdims=True
    )

    exp_logits = np.where(
        mask,
        np.exp(
            logits - row_max
        ),
        0.0
    )

    denominator = exp_logits.sum(
        axis=1,
        keepdims=True
    )

    return np.divide(
        exp_logits,
        denominator,
        out=np.zeros_like(exp_logits),
        where=denominator > 0
    )


# ============================================================
# CROSS ENTROPY
# ============================================================

def pred_ce(
    Y,
    P,
    mask
):

    return -np.sum(
        np.where(
            mask,
            Y * np.log(P + EPS),
            0.0
        )
    ) / len(Y)


# ============================================================
# AJUSTE TEMPERATURA
# ============================================================

def pred_fit_temperature(
    E,
    Y,
    mask
):

    def objective(log_T):

        T = np.exp(
            log_T
        )

        P = pred_energy_softmax(
            E,
            mask,
            T
        )

        return pred_ce(
            Y,
            P,
            mask
        )

    result = minimize_scalar(
        objective,
        bounds=(-8, 12),
        method="bounded"
    )

    return np.exp(
        result.x
    )


# ============================================================
# MIXED
# ============================================================

def pred_fit_mixed(
    E_ctx,
    E_th,
    Y,
    mask,
    T_ctx,
    T_th
):

    def objective(beta):

        P = pred_mixed_softmax(
            E_th,
            E_ctx,
            mask,
            beta[0],
            beta[1]
        )

        return pred_ce(
            Y,
            P,
            mask
        )

    starts = [
        [1 / T_th, 0.0],
        [0.0, 1 / T_ctx],
        [1 / T_th, 1 / T_ctx],
        [0.001, 0.001],
    ]

    candidates = [
        minimize(
            objective,
            x0=np.asarray(
                start,
                dtype=np.float64
            ),
            method="L-BFGS-B",
            bounds=[
                (0.0, None),
                (0.0, None),
            ]
        )
        for start in starts
    ]

    best = min(
        candidates,
        key=lambda x: x.fun
    )

    return (
        best.x[0],
        best.x[1]
    )


# ============================================================
# CALIBRACIÓN EN TRAIN
# ============================================================

T_ctx_pred = pred_fit_temperature(
    pred_Ectx[tr],
    pred_Y[tr],
    pred_mask[tr]
)

T_th_pred = pred_fit_temperature(
    pred_Eth[tr],
    pred_Y[tr],
    pred_mask[tr]
)

beta_th_pred, beta_ctx_pred = pred_fit_mixed(
    pred_Ectx[tr],
    pred_Eth[tr],
    pred_Y[tr],
    pred_mask[tr],
    T_ctx_pred,
    T_th_pred
)


# ============================================================
# PREDICCIONES
# ============================================================

pred_P_ctx = pred_energy_softmax(
    pred_Ectx,
    pred_mask,
    T_ctx_pred
)

pred_P_th = pred_energy_softmax(
    pred_Eth,
    pred_mask,
    T_th_pred
)

pred_P_mix = pred_mixed_softmax(
    pred_Eth,
    pred_Ectx,
    pred_mask,
    beta_th_pred,
    beta_ctx_pred
)

pred_P_persistence = pred_closure(
    pred_yp_current,
    pred_mask
)


pred_models = {
    "Persistence": pred_P_persistence,
    "Softmax_Context": pred_P_ctx,
    "Softmax_Theory": pred_P_th,
    "Softmax_Mixed": pred_P_mix,
}


print(
    f"T_context: {T_ctx_pred:.6f}"
)

print(
    f"T_theory : {T_th_pred:.6f}"
)

print(
    f"β_theory : {beta_th_pred:.12f}"
)

print(
    f"β_context: {beta_ctx_pred:.12f}"
)

T_context: 162753.710121
T_theory : 28.525916
β_theory : 0.035055882162
β_context: 0.000000000000


In [27]:
# ============================================================
# TOP 1% - DESCOMPOSICIÓN DETALLADA
# ============================================================

ANALYSIS_SPLIT = "test"

s = pred_splits[
    ANALYSIS_SPLIT
]

asset_to_idx = {
    asset: i
    for i, asset in enumerate(assets)
}

top1_idx = np.array([
    asset_to_idx[a]
    for a in top1_assets
])


# ============================================================
# CLR GLOBAL PARA DESCOMPONER AITCHISON
# ============================================================

def clr_matrix(
    X,
    mask
):

    Xs = np.where(
        mask,
        np.maximum(X, EPS),
        np.nan
    )

    logX = np.log(
        Xs
    )

    return (
        logX
        - np.nanmean(
            logX,
            axis=1,
            keepdims=True
        )
    )


Y_test = pred_Y[s]
M_test = pred_mask[s]

clr_Y = clr_matrix(
    Y_test,
    M_test
)


# ============================================================
# MÉTRICAS POR ACTIVO
# ============================================================

detail_rows = []


for model_name, P_all in pred_models.items():

    P = P_all[s]

    clr_P = clr_matrix(
        P,
        M_test
    )

    clr_sq = (
        clr_P - clr_Y
    ) ** 2

    total_clr_sq = np.nansum(
        clr_sq
    )


    for j in top1_idx:

        valid = M_test[:, j]

        if not valid.any():
            continue

        y = Y_test[
            valid,
            j
        ]

        p = P[
            valid,
            j
        ]

        m = (
            y + p
        ) / 2.0


        ce_j = np.mean(
            -y * np.log(
                p + EPS
            )
        )

        kl_j = np.mean(
            y * (
                np.log(y + EPS)
                - np.log(p + EPS)
            )
        )

        js_j = np.mean(
            0.5 * y * (
                np.log(y + EPS)
                - np.log(m + EPS)
            )
            +
            0.5 * p * (
                np.log(p + EPS)
                - np.log(m + EPS)
            )
        )

        error = (
            p - y
        )

        rmse_j = np.sqrt(
            np.mean(
                error ** 2
            )
        )

        mae_j = np.mean(
            np.abs(error)
        )

        mean_y_test = np.mean(
            y
        )

        nmae_j = (
            mae_j
            / mean_y_test
            if mean_y_test > EPS
            else np.nan
        )

        bias_j = np.mean(
            error
        )

        clr_rmse_j = np.sqrt(
            np.nanmean(
                clr_sq[:, j]
            )
        )

        aitchison_share = (
            100
            * np.nansum(
                clr_sq[:, j]
            )
            / total_clr_sq
            if total_clr_sq > 0
            else np.nan
        )


        detail_rows.append({
            "asset": assets[j],
            "model": model_name,
            "yp_mean_train": yp_mean_train[j],
            "yp_mean_test": mean_y_test,
            "CE_contribution": ce_j,
            "KL_contribution": kl_j,
            "JS_contribution": js_j,
            "RMSE": rmse_j,
            "MAE": mae_j,
            "NMAE": nmae_j,
            "Bias": bias_j,
            "CLR_RMSE": clr_rmse_j,
            "Aitchison_sq_share_%": aitchison_share,
        })


top1_detail = pd.DataFrame(
    detail_rows
).sort_values(
    [
        "asset",
        "model"
    ]
)

top1_detail

,asset,model,yp_mean_train,yp_mean_test,CE_contribution,KL_contribution,JS_contribution,RMSE,MAE,NMAE,Bias,CLR_RMSE,Aitchison_sq_share_%
0,BTC,Persistence,0.147572,0.183896,0.309316,0.019182,0.004610,0.085419,0.063090,0.343073,-3.046242e-05,0.561502,0.027796
6,BTC,Softmax_Context,0.147572,0.183896,1.213309,0.923174,0.058094,0.202254,0.181350,0.986157,-1.813499e-01,7.489805,0.524903
18,BTC,Softmax_Mixed,0.147572,0.183896,1.095779,0.805644,0.057981,0.202223,0.181313,0.985958,-1.813132e-01,6.739231,2.045191
12,BTC,Softmax_Theory,0.147572,0.183896,1.095779,0.805644,0.057981,0.202223,0.181313,0.985958,-1.813132e-01,6.739231,2.045191
5,DOGE,Persistence,0.045686,0.017906,0.071774,0.004319,0.000993,0.016182,0.009262,0.517226,-3.152460e-06,0.651252,0.037392
11,DOGE,Softmax_Context,0.045686,0.017906,0.129433,0.061978,0.003630,0.022120,0.015441,0.862343,-1.543876e-02,7.143754,0.477519
23,DOGE,Softmax_Mixed,0.045686,0.017906,0.106980,0.039525,0.003499,0.022065,0.015344,0.856914,-1.534060e-02,4.340135,0.848242
17,DOGE,Softmax_Theory,0.045686,0.017906,0.106980,0.039525,0.003499,0.022065,0.015344,0.856914,-1.534060e-02,4.340135,0.848242
1,ETH,Persistence,0.086714,0.092560,0.220544,0.013442,0.003175,0.055351,0.037576,0.405961,-2.561984e-05,0.545012,0.026187
7,ETH,Softmax_Context,0.086714,0.092560,0.637208,0.430107,0.027310,0.105121,0.090005,0.972395,-9.000483e-02,7.205181,0.485767


In [28]:
# ============================================================
# TOP 1% - DESCOMPOSICIÓN DETALLADA
# ============================================================

ANALYSIS_SPLIT = "test"

s = pred_splits[
    ANALYSIS_SPLIT
]

asset_to_idx = {
    asset: i
    for i, asset in enumerate(assets)
}

top1_idx = np.array([
    asset_to_idx[a]
    for a in top1_assets
])


# ============================================================
# CLR GLOBAL PARA DESCOMPONER AITCHISON
# ============================================================

def clr_matrix(
    X,
    mask
):

    Xs = np.where(
        mask,
        np.maximum(X, EPS),
        np.nan
    )

    logX = np.log(
        Xs
    )

    return (
        logX
        - np.nanmean(
            logX,
            axis=1,
            keepdims=True
        )
    )


Y_test = pred_Y[s]
M_test = pred_mask[s]

clr_Y = clr_matrix(
    Y_test,
    M_test
)


# ============================================================
# MÉTRICAS POR ACTIVO
# ============================================================

detail_rows = []


for model_name, P_all in pred_models.items():

    P = P_all[s]

    clr_P = clr_matrix(
        P,
        M_test
    )

    clr_sq = (
        clr_P - clr_Y
    ) ** 2

    total_clr_sq = np.nansum(
        clr_sq
    )


    for j in top1_idx:

        valid = M_test[:, j]

        if not valid.any():
            continue

        y = Y_test[
            valid,
            j
        ]

        p = P[
            valid,
            j
        ]

        m = (
            y + p
        ) / 2.0


        ce_j = np.mean(
            -y * np.log(
                p + EPS
            )
        )

        kl_j = np.mean(
            y * (
                np.log(y + EPS)
                - np.log(p + EPS)
            )
        )

        js_j = np.mean(
            0.5 * y * (
                np.log(y + EPS)
                - np.log(m + EPS)
            )
            +
            0.5 * p * (
                np.log(p + EPS)
                - np.log(m + EPS)
            )
        )

        error = (
            p - y
        )

        rmse_j = np.sqrt(
            np.mean(
                error ** 2
            )
        )

        mae_j = np.mean(
            np.abs(error)
        )

        mean_y_test = np.mean(
            y
        )

        nmae_j = (
            mae_j
            / mean_y_test
            if mean_y_test > EPS
            else np.nan
        )

        bias_j = np.mean(
            error
        )

        clr_rmse_j = np.sqrt(
            np.nanmean(
                clr_sq[:, j]
            )
        )

        aitchison_share = (
            100
            * np.nansum(
                clr_sq[:, j]
            )
            / total_clr_sq
            if total_clr_sq > 0
            else np.nan
        )


        detail_rows.append({
            "asset": assets[j],
            "model": model_name,
            "yp_mean_train": yp_mean_train[j],
            "yp_mean_test": mean_y_test,
            "CE_contribution": ce_j,
            "KL_contribution": kl_j,
            "JS_contribution": js_j,
            "RMSE": rmse_j,
            "MAE": mae_j,
            "NMAE": nmae_j,
            "Bias": bias_j,
            "CLR_RMSE": clr_rmse_j,
            "Aitchison_sq_share_%": aitchison_share,
        })


top1_detail = pd.DataFrame(
    detail_rows
).sort_values(
    [
        "asset",
        "model"
    ]
)

top1_detail

,asset,model,yp_mean_train,yp_mean_test,CE_contribution,KL_contribution,JS_contribution,RMSE,MAE,NMAE,Bias,CLR_RMSE,Aitchison_sq_share_%
0,BTC,Persistence,0.147572,0.183896,0.309316,0.019182,0.004610,0.085419,0.063090,0.343073,-3.046242e-05,0.561502,0.027796
6,BTC,Softmax_Context,0.147572,0.183896,1.213309,0.923174,0.058094,0.202254,0.181350,0.986157,-1.813499e-01,7.489805,0.524903
18,BTC,Softmax_Mixed,0.147572,0.183896,1.095779,0.805644,0.057981,0.202223,0.181313,0.985958,-1.813132e-01,6.739231,2.045191
12,BTC,Softmax_Theory,0.147572,0.183896,1.095779,0.805644,0.057981,0.202223,0.181313,0.985958,-1.813132e-01,6.739231,2.045191
5,DOGE,Persistence,0.045686,0.017906,0.071774,0.004319,0.000993,0.016182,0.009262,0.517226,-3.152460e-06,0.651252,0.037392
11,DOGE,Softmax_Context,0.045686,0.017906,0.129433,0.061978,0.003630,0.022120,0.015441,0.862343,-1.543876e-02,7.143754,0.477519
23,DOGE,Softmax_Mixed,0.045686,0.017906,0.106980,0.039525,0.003499,0.022065,0.015344,0.856914,-1.534060e-02,4.340135,0.848242
17,DOGE,Softmax_Theory,0.045686,0.017906,0.106980,0.039525,0.003499,0.022065,0.015344,0.856914,-1.534060e-02,4.340135,0.848242
1,ETH,Persistence,0.086714,0.092560,0.220544,0.013442,0.003175,0.055351,0.037576,0.405961,-2.561984e-05,0.545012,0.026187
7,ETH,Softmax_Context,0.086714,0.092560,0.637208,0.430107,0.027310,0.105121,0.090005,0.972395,-9.000483e-02,7.205181,0.485767


In [29]:
top1_detail[
    top1_detail["model"]
    == "Softmax_Theory"
].sort_values(
    "yp_mean_train",
    ascending=False
)

,asset,model,yp_mean_train,yp_mean_test,CE_contribution,KL_contribution,JS_contribution,RMSE,MAE,NMAE,Bias,CLR_RMSE,Aitchison_sq_share_%
12,BTC,Softmax_Theory,0.147572,0.183896,1.095779,0.805644,0.057981,0.202223,0.181313,0.985958,-0.181313,6.739231,2.045191
13,ETH,Softmax_Theory,0.086714,0.092560,0.550758,0.343657,0.027181,0.105061,0.089951,0.971815,-0.089951,6.026104,1.635259
14,USDC,Softmax_Theory,0.066578,0.100144,0.595737,0.402297,0.029980,0.140715,0.097535,0.973952,-0.097535,5.945662,1.591892
15,XRP,Softmax_Theory,0.055162,0.049506,0.296054,0.155759,0.013170,0.057308,0.046972,0.948808,-0.046972,5.410312,1.318129
16,SOL,Softmax_Theory,0.052947,0.046778,0.280757,0.145156,0.012338,0.054044,0.044301,0.947047,-0.044301,5.388092,1.307324
17,DOGE,Softmax_Theory,0.045686,0.017906,0.106980,0.039525,0.003499,0.022065,0.015344,0.856914,-0.015341,4.340135,0.848242


In [30]:
top1_detail[
    top1_detail["asset"] == "BTC"
]

,asset,model,yp_mean_train,yp_mean_test,CE_contribution,KL_contribution,JS_contribution,RMSE,MAE,NMAE,Bias,CLR_RMSE,Aitchison_sq_share_%
0,BTC,Persistence,0.147572,0.183896,0.309316,0.019182,0.004610,0.085419,0.063090,0.343073,-0.000030,0.561502,0.027796
6,BTC,Softmax_Context,0.147572,0.183896,1.213309,0.923174,0.058094,0.202254,0.181350,0.986157,-0.181350,7.489805,0.524903
18,BTC,Softmax_Mixed,0.147572,0.183896,1.095779,0.805644,0.057981,0.202223,0.181313,0.985958,-0.181313,6.739231,2.045191
12,BTC,Softmax_Theory,0.147572,0.183896,1.095779,0.805644,0.057981,0.202223,0.181313,0.985958,-0.181313,6.739231,2.045191


Agregados top 1%, top 5% y bottom 50%

In [31]:
# ============================================================
# MÉTRICAS AGREGADAS POR GRUPO
# ============================================================

GROUPS = {
    "Top_1%": top1_assets,
    "Top_5%": top5_assets,
    "Bottom_50%": bottom50_assets,
}


# ============================================================
# CIERRE INTERNO DEL GRUPO
# ============================================================

def subset_closure(
    X,
    mask
):

    z = np.where(
        mask,
        np.maximum(X, EPS),
        0.0
    )

    total = z.sum(
        axis=1,
        keepdims=True
    )

    return np.divide(
        z,
        total,
        out=np.zeros_like(z),
        where=total > EPS
    )


# ============================================================
# EVALUACIÓN DE GRUPO
# ============================================================

def evaluate_group(
    Y,
    P,
    mask,
    idx
):

    Yg_raw = Y[:, idx]
    Pg_raw = P[:, idx]
    Mg = mask[:, idx]


    valid_rows = (
        Mg.any(axis=1)
    )

    Yg_raw = Yg_raw[
        valid_rows
    ]

    Pg_raw = Pg_raw[
        valid_rows
    ]

    Mg = Mg[
        valid_rows
    ]


    # --------------------------------------------
    # MASA DEL GRUPO
    # --------------------------------------------

    target_mass = np.where(
        Mg,
        Yg_raw,
        0.0
    ).sum(
        axis=1
    )

    pred_mass = np.where(
        Mg,
        Pg_raw,
        0.0
    ).sum(
        axis=1
    )

    mass_mae = np.mean(
        np.abs(
            pred_mass
            - target_mass
        )
    )

    mass_rmse = np.sqrt(
        np.mean(
            (
                pred_mass
                - target_mass
            ) ** 2
        )
    )

    mass_bias = np.mean(
        pred_mass
        - target_mass
    )


    # --------------------------------------------
    # SIMPLEX CONDICIONAL DEL GRUPO
    # --------------------------------------------

    Yg = subset_closure(
        Yg_raw,
        Mg
    )

    Pg = subset_closure(
        Pg_raw,
        Mg
    )


    # --------------------------------------------
    # CE
    # --------------------------------------------

    CE = -np.mean(
        np.sum(
            np.where(
                Mg,
                Yg * np.log(
                    Pg + EPS
                ),
                0.0
            ),
            axis=1
        )
    )


    # --------------------------------------------
    # KL
    # --------------------------------------------

    KL = np.mean(
        np.sum(
            np.where(
                Mg,
                Yg * (
                    np.log(Yg + EPS)
                    - np.log(Pg + EPS)
                ),
                0.0
            ),
            axis=1
        )
    )


    # --------------------------------------------
    # JS
    # --------------------------------------------

    M = (
        Yg + Pg
    ) / 2

    JS = 0.5 * np.mean(
        np.sum(
            np.where(
                Mg,
                Yg * (
                    np.log(Yg + EPS)
                    - np.log(M + EPS)
                ),
                0.0
            ),
            axis=1
        )
    ) + 0.5 * np.mean(
        np.sum(
            np.where(
                Mg,
                Pg * (
                    np.log(Pg + EPS)
                    - np.log(M + EPS)
                ),
                0.0
            ),
            axis=1
        )
    )


    # --------------------------------------------
    # RMSE / MAE
    # --------------------------------------------

    diff = np.where(
        Mg,
        Pg - Yg,
        np.nan
    )

    RMSE = np.sqrt(
        np.nanmean(
            diff ** 2
        )
    )

    MAE = np.nanmean(
        np.abs(diff)
    )


    # --------------------------------------------
    # AITCHISON
    # --------------------------------------------

    logY = np.where(
        Mg,
        np.log(Yg + EPS),
        np.nan
    )

    logP = np.where(
        Mg,
        np.log(Pg + EPS),
        np.nan
    )

    clrY = (
        logY
        - np.nanmean(
            logY,
            axis=1,
            keepdims=True
        )
    )

    clrP = (
        logP
        - np.nanmean(
            logP,
            axis=1,
            keepdims=True
        )
    )

    Aitchison = np.nanmean(
        np.sqrt(
            np.nansum(
                (
                    clrY - clrP
                ) ** 2,
                axis=1
            )
        )
    )


    # --------------------------------------------
    # NMAE
    # --------------------------------------------

    nmae_values = []

    for j in range(
        Yg.shape[1]
    ):

        valid = Mg[:, j]

        if valid.any():

            mae_j = np.mean(
                np.abs(
                    Pg[valid, j]
                    - Yg[valid, j]
                )
            )

            mean_j = np.mean(
                Yg[valid, j]
            )

            if mean_j > EPS:

                nmae_values.append(
                    mae_j / mean_j
                )


    return {
        "n_assets": len(idx),

        "target_mass_mean":
            np.mean(target_mass),

        "pred_mass_mean":
            np.mean(pred_mass),

        "mass_MAE":
            mass_mae,

        "mass_RMSE":
            mass_rmse,

        "mass_bias":
            mass_bias,

        "CE":
            CE,

        "KL":
            KL,

        "JS":
            JS,

        "Aitchison":
            Aitchison,

        "RMSE":
            RMSE,

        "MAE":
            MAE,

        "NMAE_mean":
            np.mean(nmae_values),

        "NMAE_median":
            np.median(nmae_values),
    }


# ============================================================
# TEST
# ============================================================

s = pred_splits[
    "test"
]

group_results = []


for group_name, group_assets in GROUPS.items():

    idx = np.array([
        asset_to_idx[a]
        for a in group_assets
    ])


    for model_name, P_all in pred_models.items():

        metrics = evaluate_group(
            pred_Y[s],
            P_all[s],
            pred_mask[s],
            idx
        )

        group_results.append({
            "group": group_name,
            "model": model_name,
            **metrics
        })


group_results = pd.DataFrame(
    group_results
)

group_results

,group,model,n_assets,target_mass_mean,pred_mass_mean,mass_MAE,mass_RMSE,mass_bias,CE,KL,JS,Aitchison,RMSE,MAE,NMAE_mean,NMAE_median
0,Top_1%,Persistence,6,0.49079,0.490724,0.087921,0.114252,-0.000067,1.633814,0.181158,0.042182,1.350749,0.114864,0.072390,0.469612,0.452535
1,Top_1%,Softmax_Context,6,0.49079,0.015287,0.475503,0.492054,-0.475503,2.564408,1.111751,0.096903,6.092730,0.150907,0.117188,1.107869,0.701441
2,Top_1%,Softmax_Theory,6,0.49079,0.015377,0.475413,0.491946,-0.475413,1.786678,0.334021,0.083513,2.199234,0.145219,0.112075,1.074084,0.651794
3,Top_1%,Softmax_Mixed,6,0.49079,0.015377,0.475413,0.491946,-0.475413,1.786678,0.334021,0.083513,2.199234,0.145219,0.112074,1.074084,0.651794
4,Top_5%,Persistence,26,0.71880,0.718682,0.059307,0.077919,-0.000118,2.517180,0.272950,0.061026,3.856173,0.051034,0.020060,0.616697,0.642158
5,Top_5%,Softmax_Context,26,0.71880,0.066106,0.652694,0.659886,-0.652694,4.104144,1.859914,0.237440,20.866864,0.073781,0.044379,4.502411,2.354857
6,Top_5%,Softmax_Theory,26,0.71880,0.065355,0.653445,0.660597,-0.653445,3.240169,0.995939,0.224640,8.223590,0.072947,0.043440,4.420883,2.189211
7,Top_5%,Softmax_Mixed,26,0.71880,0.065355,0.653445,0.660597,-0.653445,3.240169,0.995939,0.224640,8.223590,0.072947,0.043440,4.420883,2.189211
8,Bottom_50%,Persistence,255,0.05724,0.057324,0.017060,0.024286,0.000085,3.887468,0.569476,0.110226,23.723859,0.016762,0.004800,0.902425,0.908134
9,Bottom_50%,Softmax_Context,255,0.05724,0.366033,0.308793,0.310419,0.308793,6.172738,2.854746,0.315946,64.097298,0.025229,0.009075,2.542586,1.988033


In [32]:
# ============================================================
# MÉTRICAS AGREGADAS POR GRUPO
# ============================================================

GROUPS = {
    "Top_1%": top1_assets,
    "Top_5%": top5_assets,
    "Bottom_50%": bottom50_assets,
}


# ============================================================
# CIERRE INTERNO DEL GRUPO
# ============================================================

def subset_closure(
    X,
    mask
):

    z = np.where(
        mask,
        np.maximum(X, EPS),
        0.0
    )

    total = z.sum(
        axis=1,
        keepdims=True
    )

    return np.divide(
        z,
        total,
        out=np.zeros_like(z),
        where=total > EPS
    )


# ============================================================
# EVALUACIÓN DE GRUPO
# ============================================================

def evaluate_group(
    Y,
    P,
    mask,
    idx
):

    Yg_raw = Y[:, idx]
    Pg_raw = P[:, idx]
    Mg = mask[:, idx]


    valid_rows = (
        Mg.any(axis=1)
    )

    Yg_raw = Yg_raw[
        valid_rows
    ]

    Pg_raw = Pg_raw[
        valid_rows
    ]

    Mg = Mg[
        valid_rows
    ]


    # --------------------------------------------
    # MASA DEL GRUPO
    # --------------------------------------------

    target_mass = np.where(
        Mg,
        Yg_raw,
        0.0
    ).sum(
        axis=1
    )

    pred_mass = np.where(
        Mg,
        Pg_raw,
        0.0
    ).sum(
        axis=1
    )

    mass_mae = np.mean(
        np.abs(
            pred_mass
            - target_mass
        )
    )

    mass_rmse = np.sqrt(
        np.mean(
            (
                pred_mass
                - target_mass
            ) ** 2
        )
    )

    mass_bias = np.mean(
        pred_mass
        - target_mass
    )


    # --------------------------------------------
    # SIMPLEX CONDICIONAL DEL GRUPO
    # --------------------------------------------

    Yg = subset_closure(
        Yg_raw,
        Mg
    )

    Pg = subset_closure(
        Pg_raw,
        Mg
    )


    # --------------------------------------------
    # CE
    # --------------------------------------------

    CE = -np.mean(
        np.sum(
            np.where(
                Mg,
                Yg * np.log(
                    Pg + EPS
                ),
                0.0
            ),
            axis=1
        )
    )


    # --------------------------------------------
    # KL
    # --------------------------------------------

    KL = np.mean(
        np.sum(
            np.where(
                Mg,
                Yg * (
                    np.log(Yg + EPS)
                    - np.log(Pg + EPS)
                ),
                0.0
            ),
            axis=1
        )
    )


    # --------------------------------------------
    # JS
    # --------------------------------------------

    M = (
        Yg + Pg
    ) / 2

    JS = 0.5 * np.mean(
        np.sum(
            np.where(
                Mg,
                Yg * (
                    np.log(Yg + EPS)
                    - np.log(M + EPS)
                ),
                0.0
            ),
            axis=1
        )
    ) + 0.5 * np.mean(
        np.sum(
            np.where(
                Mg,
                Pg * (
                    np.log(Pg + EPS)
                    - np.log(M + EPS)
                ),
                0.0
            ),
            axis=1
        )
    )


    # --------------------------------------------
    # RMSE / MAE
    # --------------------------------------------

    diff = np.where(
        Mg,
        Pg - Yg,
        np.nan
    )

    RMSE = np.sqrt(
        np.nanmean(
            diff ** 2
        )
    )

    MAE = np.nanmean(
        np.abs(diff)
    )


    # --------------------------------------------
    # AITCHISON
    # --------------------------------------------

    logY = np.where(
        Mg,
        np.log(Yg + EPS),
        np.nan
    )

    logP = np.where(
        Mg,
        np.log(Pg + EPS),
        np.nan
    )

    clrY = (
        logY
        - np.nanmean(
            logY,
            axis=1,
            keepdims=True
        )
    )

    clrP = (
        logP
        - np.nanmean(
            logP,
            axis=1,
            keepdims=True
        )
    )

    Aitchison = np.nanmean(
        np.sqrt(
            np.nansum(
                (
                    clrY - clrP
                ) ** 2,
                axis=1
            )
        )
    )


    # --------------------------------------------
    # NMAE
    # --------------------------------------------

    nmae_values = []

    for j in range(
        Yg.shape[1]
    ):

        valid = Mg[:, j]

        if valid.any():

            mae_j = np.mean(
                np.abs(
                    Pg[valid, j]
                    - Yg[valid, j]
                )
            )

            mean_j = np.mean(
                Yg[valid, j]
            )

            if mean_j > EPS:

                nmae_values.append(
                    mae_j / mean_j
                )


    return {
        "n_assets": len(idx),

        "target_mass_mean":
            np.mean(target_mass),

        "pred_mass_mean":
            np.mean(pred_mass),

        "mass_MAE":
            mass_mae,

        "mass_RMSE":
            mass_rmse,

        "mass_bias":
            mass_bias,

        "CE":
            CE,

        "KL":
            KL,

        "JS":
            JS,

        "Aitchison":
            Aitchison,

        "RMSE":
            RMSE,

        "MAE":
            MAE,

        "NMAE_mean":
            np.mean(nmae_values),

        "NMAE_median":
            np.median(nmae_values),
    }


# ============================================================
# TEST
# ============================================================

s = pred_splits[
    "test"
]

group_results = []


for group_name, group_assets in GROUPS.items():

    idx = np.array([
        asset_to_idx[a]
        for a in group_assets
    ])


    for model_name, P_all in pred_models.items():

        metrics = evaluate_group(
            pred_Y[s],
            P_all[s],
            pred_mask[s],
            idx
        )

        group_results.append({
            "group": group_name,
            "model": model_name,
            **metrics
        })


group_results = pd.DataFrame(
    group_results
)

group_results

,group,model,n_assets,target_mass_mean,pred_mass_mean,mass_MAE,mass_RMSE,mass_bias,CE,KL,JS,Aitchison,RMSE,MAE,NMAE_mean,NMAE_median
0,Top_1%,Persistence,6,0.49079,0.490724,0.087921,0.114252,-0.000067,1.633814,0.181158,0.042182,1.350749,0.114864,0.072390,0.469612,0.452535
1,Top_1%,Softmax_Context,6,0.49079,0.015287,0.475503,0.492054,-0.475503,2.564408,1.111751,0.096903,6.092730,0.150907,0.117188,1.107869,0.701441
2,Top_1%,Softmax_Theory,6,0.49079,0.015377,0.475413,0.491946,-0.475413,1.786678,0.334021,0.083513,2.199234,0.145219,0.112075,1.074084,0.651794
3,Top_1%,Softmax_Mixed,6,0.49079,0.015377,0.475413,0.491946,-0.475413,1.786678,0.334021,0.083513,2.199234,0.145219,0.112074,1.074084,0.651794
4,Top_5%,Persistence,26,0.71880,0.718682,0.059307,0.077919,-0.000118,2.517180,0.272950,0.061026,3.856173,0.051034,0.020060,0.616697,0.642158
5,Top_5%,Softmax_Context,26,0.71880,0.066106,0.652694,0.659886,-0.652694,4.104144,1.859914,0.237440,20.866864,0.073781,0.044379,4.502411,2.354857
6,Top_5%,Softmax_Theory,26,0.71880,0.065355,0.653445,0.660597,-0.653445,3.240169,0.995939,0.224640,8.223590,0.072947,0.043440,4.420883,2.189211
7,Top_5%,Softmax_Mixed,26,0.71880,0.065355,0.653445,0.660597,-0.653445,3.240169,0.995939,0.224640,8.223590,0.072947,0.043440,4.420883,2.189211
8,Bottom_50%,Persistence,255,0.05724,0.057324,0.017060,0.024286,0.000085,3.887468,0.569476,0.110226,23.723859,0.016762,0.004800,0.902425,0.908134
9,Bottom_50%,Softmax_Context,255,0.05724,0.366033,0.308793,0.310419,0.308793,6.172738,2.854746,0.315946,64.097298,0.025229,0.009075,2.542586,1.988033


A4 — Diagnóstico por concentración. La descomposición del desempeño por niveles de masa revela que el principal fallo del softmax energético puro no reside en la geometría relativa de los activos, sino en la asignación de masa entre estratos del simplex. En Test, el Top 1% concentra aproximadamente 49.1% de la masa observada, mientras Softmax_Theory le asigna solo 1.54%; simultáneamente, el Bottom 50% recibe aproximadamente 36.7% frente a un valor observado de 5.72%. Sin embargo, una vez condicionada la distribución dentro de cada grupo, E_theory supera consistentemente a E_context: reduce la distancia de Aitchison aproximadamente 63.9% en el Top 1%, 60.6% en el Top 5% y 57.5% en el Bottom 50%. Esto indica que la representación teórica contiene información importante sobre la geometría relativa del simplex en todos los niveles de concentración, pero el softmax uniforme no reproduce correctamente su concentración global.

### A5. Softmax energetico con medida base no uniforme

A5.1: Prior estructural

A5.2: Prior dinamico

In [33]:
# ============================================================
# A5 - SOFTMAX ENERGÉTICO CON MEDIDA BASE NO UNIFORME
# ============================================================

import numpy as np
import pandas as pd

from scipy.optimize import minimize, minimize_scalar


# ============================================================
# CONFIGURACIÓN
# ============================================================

EPS = 1e-12

TRAIN_FRAC = 0.70
VAL_FRAC = 0.15

FORBIDDEN_FEATURES = (
    "FLOW_TOTAL_AUDIT",
    "TURNOVER_PER_TRADE_AUDIT",
)


# ============================================================
# VALIDACIÓN CTX vs THEORY
# ============================================================

assert len(df1_ctx) == len(df1_th)

assert df1_ctx["bar_end"].equals(
    df1_th["bar_end"]
)


# ============================================================
# COLUMNAS UTILIZADAS
# ============================================================

ctx_cols = [
    f"{a}__E_context"
    for a in assets
]

th_cols = [
    f"{a}__E_theory"
    for a in assets
]

yp_cols = [
    f"{a}{cfg.yp_suffix}"
    for a in assets
]

active_cols = [
    f"{a}{cfg.active_suffix}"
    for a in assets
]

used_columns = (
    ctx_cols
    + th_cols
    + yp_cols
    + active_cols
)

assert not any(
    forbidden in col
    for forbidden in FORBIDDEN_FEATURES
    for col in used_columns
)

print("Variables AUDIT excluidas correctamente.")


# ============================================================
# CARGA EFICIENTE
# ============================================================

E_ctx = df1_ctx.loc[
    :,
    ctx_cols
].to_numpy(
    dtype=np.float32,
    copy=True
)

E_th = df1_th.loc[
    :,
    th_cols
].to_numpy(
    dtype=np.float32,
    copy=True
)

yp_t = df1_ctx.loc[
    :,
    yp_cols
].to_numpy(
    dtype=np.float32,
    copy=True
)

A_t = df1_ctx.loc[
    :,
    active_cols
].to_numpy(
    dtype=np.float32,
    copy=True
)


# ============================================================
# TARGET t+1
# ============================================================

Y_raw = np.full_like(
    yp_t,
    np.nan
)

Y_raw[:-1] = yp_t[1:]


# ============================================================
# SOPORTE COMÚN
# ============================================================

mask = (
    (A_t > 0)
    & np.isfinite(E_ctx)
    & np.isfinite(E_th)
    & np.isfinite(yp_t)
    & np.isfinite(Y_raw)
)


# ============================================================
# CIERRE AL SIMPLEX
# ============================================================

def closure(
    x,
    mask,
    eps=EPS
):

    z = np.where(
        mask,
        np.maximum(x, 0.0),
        0.0
    )

    total = z.sum(
        axis=1,
        keepdims=True
    )

    return np.divide(
        z,
        total,
        out=np.zeros_like(
            z,
            dtype=np.float64
        ),
        where=total > eps
    )


Y = closure(
    Y_raw,
    mask
)

P_dynamic_prior = closure(
    yp_t,
    mask
)


# ============================================================
# FILAS VÁLIDAS
# ============================================================

valid_rows = (
    (mask.sum(axis=1) >= 2)
    & (Y.sum(axis=1) > 1 - 1e-8)
    & (P_dynamic_prior.sum(axis=1) > 1 - 1e-8)
)

E_ctx = E_ctx[valid_rows]
E_th = E_th[valid_rows]

Y = Y[valid_rows]

mask = mask[valid_rows]

P_dynamic_prior = P_dynamic_prior[
    valid_rows
]


# ============================================================
# SPLIT TEMPORAL
# ============================================================

n = len(Y)

i_train = int(
    n * TRAIN_FRAC
)

i_val = int(
    n * (TRAIN_FRAC + VAL_FRAC)
)

splits = {
    "train": slice(
        0,
        i_train
    ),

    "val": slice(
        i_train,
        i_val
    ),

    "test": slice(
        i_val,
        n
    ),
}

tr = splits[
    "train"
]


print(
    f"Train: {i_train:,} | "
    f"Val: {i_val-i_train:,} | "
    f"Test: {n-i_val:,}"
)


# ============================================================
# A5.1 - PRIOR ESTRUCTURAL
# Exclusivamente Train
# ============================================================

structural_prior = Y[
    tr
].mean(
    axis=0
)

structural_prior = np.maximum(
    structural_prior,
    EPS
)

structural_prior /= structural_prior.sum()


print(
    "\nMasa prior estructural:",
    structural_prior.sum()
)


# ============================================================
# SOFTMAX GENERALIZADO
# p ∝ prior * exp(-βE)
# ============================================================

def prior_energy_softmax(
    prior,
    mask,
    E_ctx=None,
    E_th=None,
    beta_ctx=0.0,
    beta_th=0.0,
):

    if prior.ndim == 1:

        base = np.broadcast_to(
            prior,
            mask.shape
        )

    else:

        base = prior


    logits = np.where(
        mask,
        np.log(
            np.maximum(
                base,
                EPS
            )
        ),
        -np.inf
    )


    if E_ctx is not None:

        logits = np.where(
            mask,
            logits
            - beta_ctx * E_ctx,
            -np.inf
        )


    if E_th is not None:

        logits = np.where(
            mask,
            logits
            - beta_th * E_th,
            -np.inf
        )


    row_max = np.max(
        logits,
        axis=1,
        keepdims=True
    )

    exp_logits = np.where(
        mask,
        np.exp(
            logits - row_max
        ),
        0.0
    )

    denominator = exp_logits.sum(
        axis=1,
        keepdims=True
    )


    return np.divide(
        exp_logits,
        denominator,
        out=np.zeros_like(
            exp_logits
        ),
        where=denominator > 0
    )


# ============================================================
# CROSS ENTROPY
# ============================================================

def cross_entropy(
    Y,
    P,
    mask
):

    return -np.sum(
        np.where(
            mask,
            Y * np.log(
                P + EPS
            ),
            0.0
        )
    ) / len(Y)


# ============================================================
# AJUSTE β INDIVIDUAL
# β = 0 también es candidato válido
# ============================================================

def fit_beta(
    prior,
    E,
    Y,
    mask
):

    P0 = prior_energy_softmax(
        prior,
        mask,
        E_th=E,
        beta_th=0.0
    )

    ce0 = cross_entropy(
        Y,
        P0,
        mask
    )


    def objective(
        log_beta
    ):

        beta = np.exp(
            log_beta
        )

        P = prior_energy_softmax(
            prior,
            mask,
            E_th=E,
            beta_th=beta
        )

        return cross_entropy(
            Y,
            P,
            mask
        )


    result = minimize_scalar(
        objective,
        bounds=(-20, 10),
        method="bounded"
    )


    beta = np.exp(
        result.x
    )


    if ce0 <= result.fun:

        return 0.0, ce0


    return beta, result.fun


# ============================================================
# AJUSTE MIXTO ROBUSTO
# ============================================================

def fit_mixed(
    prior,
    E_ctx,
    E_th,
    Y,
    mask,
    beta_ctx_pure,
    beta_th_pure,
):

    def objective(
        theta
    ):

        beta_th = np.exp(
            theta[0]
        )

        beta_ctx = np.exp(
            theta[1]
        )

        P = prior_energy_softmax(
            prior,
            mask,
            E_ctx=E_ctx,
            E_th=E_th,
            beta_ctx=beta_ctx,
            beta_th=beta_th,
        )

        return cross_entropy(
            Y,
            P,
            mask
        )


    candidates = []


    # --------------------------------------------
    # Caso β = 0, 0
    # --------------------------------------------

    P0 = prior_energy_softmax(
        prior,
        mask
    )

    candidates.append((
        cross_entropy(
            Y,
            P0,
            mask
        ),
        0.0,
        0.0
    ))


    # --------------------------------------------
    # Context puro
    # --------------------------------------------

    P_ctx = prior_energy_softmax(
        prior,
        mask,
        E_ctx=E_ctx,
        beta_ctx=beta_ctx_pure
    )

    candidates.append((
        cross_entropy(
            Y,
            P_ctx,
            mask
        ),
        0.0,
        beta_ctx_pure
    ))


    # --------------------------------------------
    # Theory puro
    # --------------------------------------------

    P_th = prior_energy_softmax(
        prior,
        mask,
        E_th=E_th,
        beta_th=beta_th_pure
    )

    candidates.append((
        cross_entropy(
            Y,
            P_th,
            mask
        ),
        beta_th_pure,
        0.0
    ))


    # --------------------------------------------
    # Interior mixto
    # --------------------------------------------

    starts = [
        [
            max(
                beta_th_pure,
                1e-8
            ),
            max(
                beta_ctx_pure,
                1e-8
            ),
        ],

        [
            max(
                beta_th_pure,
                1e-8
            ),
            1e-8,
        ],

        [
            1e-8,
            max(
                beta_ctx_pure,
                1e-8
            ),
        ],

        [
            1e-3,
            1e-3,
        ],
    ]


    for start in starts:

        result = minimize(
            objective,
            x0=np.log(
                start
            ),
            method="L-BFGS-B",
            bounds=[
                (-20, 10),
                (-20, 10),
            ]
        )

        candidates.append((
            result.fun,
            np.exp(
                result.x[0]
            ),
            np.exp(
                result.x[1]
            ),
        ))


    best = min(
        candidates,
        key=lambda x: x[0]
    )


    return (
        best[1],
        best[2],
        best[0],
    )


# ============================================================
# A5.1 - PRIOR ESTRUCTURAL
# ============================================================

beta_ctx_struct, _ = fit_beta(
    structural_prior,
    E_ctx[tr],
    Y[tr],
    mask[tr]
)

beta_th_struct, _ = fit_beta(
    structural_prior,
    E_th[tr],
    Y[tr],
    mask[tr]
)

(
    beta_th_struct_mix,
    beta_ctx_struct_mix,
    ce_struct_mix
) = fit_mixed(
    structural_prior,
    E_ctx[tr],
    E_th[tr],
    Y[tr],
    mask[tr],
    beta_ctx_struct,
    beta_th_struct,
)


# ============================================================
# A5.2 - PRIOR DINÁMICO
# yp(t)
# ============================================================

beta_ctx_dyn, _ = fit_beta(
    P_dynamic_prior[tr],
    E_ctx[tr],
    Y[tr],
    mask[tr]
)

beta_th_dyn, _ = fit_beta(
    P_dynamic_prior[tr],
    E_th[tr],
    Y[tr],
    mask[tr]
)

(
    beta_th_dyn_mix,
    beta_ctx_dyn_mix,
    ce_dyn_mix
) = fit_mixed(
    P_dynamic_prior[tr],
    E_ctx[tr],
    E_th[tr],
    Y[tr],
    mask[tr],
    beta_ctx_dyn,
    beta_th_dyn,
)


# ============================================================
# PARÁMETROS
# ============================================================

params = pd.DataFrame([
    {
        "model": "Structural_Context",
        "beta_theory": 0.0,
        "beta_context": beta_ctx_struct,
    },

    {
        "model": "Structural_Theory",
        "beta_theory": beta_th_struct,
        "beta_context": 0.0,
    },

    {
        "model": "Structural_Mixed",
        "beta_theory": beta_th_struct_mix,
        "beta_context": beta_ctx_struct_mix,
    },

    {
        "model": "Dynamic_Context",
        "beta_theory": 0.0,
        "beta_context": beta_ctx_dyn,
    },

    {
        "model": "Dynamic_Theory",
        "beta_theory": beta_th_dyn,
        "beta_context": 0.0,
    },

    {
        "model": "Dynamic_Mixed",
        "beta_theory": beta_th_dyn_mix,
        "beta_context": beta_ctx_dyn_mix,
    },
])

params["lambda_theory"] = np.divide(
    params["beta_theory"],
    (
        params["beta_theory"]
        + params["beta_context"]
    ),
    out=np.full(
        len(params),
        np.nan
    ),
    where=(
        params["beta_theory"]
        + params["beta_context"]
    ) > 0
)

params["lambda_context"] = (
    1
    - params["lambda_theory"]
)

print("\nPARÁMETROS")
display(params)


# ============================================================
# PREDICCIONES A5.1
# ============================================================

P_structural = prior_energy_softmax(
    structural_prior,
    mask
)

P_struct_ctx = prior_energy_softmax(
    structural_prior,
    mask,
    E_ctx=E_ctx,
    beta_ctx=beta_ctx_struct
)

P_struct_th = prior_energy_softmax(
    structural_prior,
    mask,
    E_th=E_th,
    beta_th=beta_th_struct
)

P_struct_mix = prior_energy_softmax(
    structural_prior,
    mask,
    E_ctx=E_ctx,
    E_th=E_th,
    beta_ctx=beta_ctx_struct_mix,
    beta_th=beta_th_struct_mix,
)


# ============================================================
# PREDICCIONES A5.2
# ============================================================

P_persistence = prior_energy_softmax(
    P_dynamic_prior,
    mask
)

P_dyn_ctx = prior_energy_softmax(
    P_dynamic_prior,
    mask,
    E_ctx=E_ctx,
    beta_ctx=beta_ctx_dyn
)

P_dyn_th = prior_energy_softmax(
    P_dynamic_prior,
    mask,
    E_th=E_th,
    beta_th=beta_th_dyn
)

P_dyn_mix = prior_energy_softmax(
    P_dynamic_prior,
    mask,
    E_ctx=E_ctx,
    E_th=E_th,
    beta_ctx=beta_ctx_dyn_mix,
    beta_th=beta_th_dyn_mix,
)


# ============================================================
# MÉTRICAS
# ============================================================

def evaluate(
    Y,
    P,
    mask
):

    CE = cross_entropy(
        Y,
        P,
        mask
    )


    # --------------------------------------------
    # KL
    # --------------------------------------------

    KL = np.mean(
        np.sum(
            np.where(
                mask,
                Y * (
                    np.log(Y + EPS)
                    - np.log(P + EPS)
                ),
                0.0
            ),
            axis=1
        )
    )


    # --------------------------------------------
    # Jensen-Shannon
    # --------------------------------------------

    M = (
        Y + P
    ) / 2.0

    JS = 0.5 * np.mean(
        np.sum(
            np.where(
                mask,
                Y * (
                    np.log(Y + EPS)
                    - np.log(M + EPS)
                ),
                0.0
            ),
            axis=1
        )
    ) + 0.5 * np.mean(
        np.sum(
            np.where(
                mask,
                P * (
                    np.log(P + EPS)
                    - np.log(M + EPS)
                ),
                0.0
            ),
            axis=1
        )
    )


    # --------------------------------------------
    # RMSE / MAE
    # --------------------------------------------

    diff = np.where(
        mask,
        P - Y,
        np.nan
    )

    RMSE = np.sqrt(
        np.nanmean(
            diff ** 2
        )
    )

    MAE = np.nanmean(
        np.abs(diff)
    )


    # --------------------------------------------
    # NMAE
    # --------------------------------------------

    nmae = []

    for j in range(
        Y.shape[1]
    ):

        valid = mask[:, j]

        if not valid.any():

            continue

        mae_j = np.mean(
            np.abs(
                P[valid, j]
                - Y[valid, j]
            )
        )

        mean_y = np.mean(
            Y[valid, j]
        )

        if mean_y > EPS:

            nmae.append(
                mae_j / mean_y
            )


    # --------------------------------------------
    # Aitchison
    # --------------------------------------------

    logY = np.where(
        mask,
        np.log(
            Y + EPS
        ),
        np.nan
    )

    logP = np.where(
        mask,
        np.log(
            P + EPS
        ),
        np.nan
    )

    clrY = (
        logY
        - np.nanmean(
            logY,
            axis=1,
            keepdims=True
        )
    )

    clrP = (
        logP
        - np.nanmean(
            logP,
            axis=1,
            keepdims=True
        )
    )

    Aitchison = np.nanmean(
        np.sqrt(
            np.nansum(
                (
                    clrY
                    - clrP
                ) ** 2,
                axis=1
            )
        )
    )


    return {
        "CE": CE,
        "KL": KL,
        "JS": JS,
        "Aitchison": Aitchison,
        "RMSE": RMSE,
        "MAE": MAE,
        "NMAE_mean": np.mean(
            nmae
        ),
        "NMAE_median": np.median(
            nmae
        ),
    }


# ============================================================
# MODELOS
# ============================================================

models = {
    "Structural_Prior": P_structural,
    "Structural_Context": P_struct_ctx,
    "Structural_Theory": P_struct_th,
    "Structural_Mixed": P_struct_mix,

    "Persistence": P_persistence,
    "Dynamic_Context": P_dyn_ctx,
    "Dynamic_Theory": P_dyn_th,
    "Dynamic_Mixed": P_dyn_mix,
}


# ============================================================
# EVALUACIÓN TRAIN / VAL / TEST
# ============================================================

results_A5 = []


for split_name in (
    "train",
    "val",
    "test",
):

    s = splits[
        split_name
    ]


    for model_name, P in models.items():

        metrics = evaluate(
            Y[s],
            P[s],
            mask[s]
        )

        results_A5.append({
            "split": split_name,
            "model": model_name,
            **metrics
        })


results_A5 = pd.DataFrame(
    results_A5
)


# ============================================================
# RESULTADOS
# ============================================================

display(
    results_A5
    .sort_values(
        [
            "split",
            "CE"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# TEST ÚNICAMENTE
# ============================================================

print("\nTEST")

display(
    results_A5[
        results_A5["split"]
        == "test"
    ]
    .sort_values(
        "CE"
    )
    .reset_index(
        drop=True
    )
)

Variables AUDIT excluidas correctamente.
Train: 9,869 | Val: 2,115 | Test: 2,115

Masa prior estructural: 1.0

PARÁMETROS


,model,beta_theory,beta_context,lambda_theory,lambda_context
0,Structural_Context,0.000000,0.0,NaN,NaN
1,Structural_Theory,0.005732,0.0,1.0,0.0
2,Structural_Mixed,0.005732,0.0,1.0,0.0
3,Dynamic_Context,0.000000,0.0,NaN,NaN
4,Dynamic_Theory,0.000000,0.0,NaN,NaN
5,Dynamic_Mixed,0.000000,0.0,NaN,NaN


,split,model,CE,KL,JS,Aitchison,RMSE,MAE,NMAE_mean,NMAE_median
0,test,Persistence,3.760318,0.353298,0.075204,32.192875,0.010288,0.001446,0.855169,0.850561
1,test,Dynamic_Context,3.760318,0.353298,0.075204,32.192875,0.010288,0.001446,0.855169,0.850561
2,test,Dynamic_Theory,3.760318,0.353298,0.075204,32.192875,0.010288,0.001446,0.855169,0.850561
3,test,Dynamic_Mixed,3.760318,0.353298,0.075204,32.192875,0.010288,0.001446,0.855169,0.850561
4,test,Structural_Theory,4.423474,1.016454,0.122920,67.709242,0.010328,0.001893,1.503382,1.226525
5,test,Structural_Mixed,4.423474,1.016454,0.122920,67.709242,0.010328,0.001893,1.503382,1.226525
6,test,Structural_Prior,4.424466,1.017446,0.123207,67.741015,0.010342,0.001896,1.510476,1.231785
7,test,Structural_Context,4.424466,1.017446,0.123207,67.741015,0.010342,0.001896,1.510476,1.231785
8,train,Persistence,3.949581,0.253080,0.055604,20.838418,0.007584,0.001267,0.692227,0.687670
9,train,Dynamic_Context,3.949581,0.253080,0.055604,20.838418,0.007584,0.001267,0.692227,0.687670



TEST


,split,model,CE,KL,JS,Aitchison,RMSE,MAE,NMAE_mean,NMAE_median
0,test,Dynamic_Mixed,3.760318,0.353298,0.075204,32.192875,0.010288,0.001446,0.855169,0.850561
1,test,Dynamic_Theory,3.760318,0.353298,0.075204,32.192875,0.010288,0.001446,0.855169,0.850561
2,test,Dynamic_Context,3.760318,0.353298,0.075204,32.192875,0.010288,0.001446,0.855169,0.850561
3,test,Persistence,3.760318,0.353298,0.075204,32.192875,0.010288,0.001446,0.855169,0.850561
4,test,Structural_Theory,4.423474,1.016454,0.122920,67.709242,0.010328,0.001893,1.503382,1.226525
5,test,Structural_Mixed,4.423474,1.016454,0.122920,67.709242,0.010328,0.001893,1.503382,1.226525
6,test,Structural_Prior,4.424466,1.017446,0.123207,67.741015,0.010342,0.001896,1.510476,1.231785
7,test,Structural_Context,4.424466,1.017446,0.123207,67.741015,0.010342,0.001896,1.510476,1.231785


A5 — Softmax energético con medida base no uniforme. La introducción de una medida base confirma que el principal problema del softmax energético uniforme era la representación del nivel de concentración del simplex. Con un prior estructural estimado exclusivamente sobre Train, E_theory recibe un coeficiente positivo (\(\beta=0.005732\)) y mejora consistentemente, aunque de manera muy pequeña, todas las métricas respecto del prior sin energía; E_context, en cambio, recibe coeficiente cero y no aporta mejora. Sin embargo, al utilizar el simplex contemporáneo \(yp_t\) como medida base dinámica, los coeficientes óptimos de Theory y Context colapsan exactamente a cero. En consecuencia, Dynamic_Theory, Dynamic_Context y Dynamic_Mixed reproducen exactamente al benchmark de Persistence. A horizonte de cinco minutos, la energía contemporánea no contiene información incremental suficiente para mejorar \(yp_t\) como predictor de \(yp_{t+1}\) bajo esta parametrización multiplicativa de Gibbs.

## Conclusiones Experimento A